# InceptionResNetV2 — reprodução e template experimental

Este notebook implementa duas abordagens para o conjunto de imagens da poça de fusão dos nove experimentos de PTA-DED.

1. **Metodologia `_old`:** reprodução das decisões empregadas por Fernandes Balbino na extração de *features*, rotulagem por interpolação, geração temporal e regressão, com o objetivo de obter resultados próximos aos reportados.
2. **Metodologia `_new`:** procedimento experimental corrigido, sem vazamento entre treino e teste e preparado como *template* para a comparação posterior com ResNet50, DenseNet169 e EfficientNet-B0.

## Parte 1 — Reprodução da metodologia `_old`

Nesta primeira parte, as decisões do código de referência serão preservadas e documentadas com o sufixo `_old`. As correções serão implementadas separadamente na Parte 2 e identificadas pelo sufixo `_new`.

### 1. Configuração do ambiente e dos caminhos

Esta etapa monta o Google Drive e valida a localização do conjunto de imagens. Nenhuma imagem é processada aqui.

In [ ]:
from google.colab import drive
from pathlib import Path

# Monta o Google Drive na sessão do Colab.
drive.mount("/content/drive")

# Caminhos permanentes no Google Drive.
PROJECT_ROOT = Path("/content/drive/MyDrive/TCC")
DATA_DIR = PROJECT_ROOT / "Dados Experimentos"
ZIP_PATH = DATA_DIR / "experimentos_taguchi_400p_cropped_images.zip"
OUTPUT_DIR = PROJECT_ROOT / "resultados" / "inceptionresnetv2"

# Caminhos temporários da sessão do Colab.
LOCAL_ROOT = Path("/content/tcc_inceptionresnetv2")
DATASET_ROOT = LOCAL_ROOT / "experimentos_taguchi_400p_cropped_images"

print(f"Projeto: {PROJECT_ROOT}")
print(f"Arquivo de imagens: {ZIP_PATH}")
print(f"ZIP encontrado: {ZIP_PATH.exists()}")
print(f"Saídas da InceptionResNetV2: {OUTPUT_DIR}")

if not ZIP_PATH.exists():
    raise FileNotFoundError(f"Conjunto de imagens não encontrado em: {ZIP_PATH}")

### 2. Preparação e validação do conjunto de imagens

O arquivo compactado é extraído para o armazenamento temporário da sessão, evitando a leitura de milhares de imagens diretamente pelo Google Drive. Em seguida, são verificadas as quantidades e dimensões das imagens de L1–L9.

In [ ]:
import subprocess

LOCAL_ROOT.mkdir(parents=True, exist_ok=True)

if DATASET_ROOT.exists():
    print(f"Dados já extraídos em: {DATASET_ROOT}")
else:
    print("Extraindo o conjunto de imagens para o armazenamento temporário...")
    subprocess.run(
        ["unzip", "-q", str(ZIP_PATH), "-d", str(LOCAL_ROOT)],
        check=True,
    )
    print("Extração concluída.")

if not DATASET_ROOT.exists():
    raise FileNotFoundError(
        f"A pasta esperada não foi encontrada após a extração: {DATASET_ROOT}"
    )

print(f"Diretório do conjunto: {DATASET_ROOT}")

#### Verificação da integridade do conjunto

O código a seguir percorre as pastas dos experimentos L1 a L9 e ordena as imagens numericamente pelo identificador do *frame*. Para cada experimento, ele compara a quantidade encontrada com a quantidade esperada e abre a primeira imagem para verificar se o arquivo está no modo RGB e possui dimensões de 400 × 400 pixels. Ao final, os resultados são organizados em uma tabela, o total de imagens é calculado e uma exceção é gerada caso alguma divergência seja identificada.

In [ ]:
import pandas as pd
from PIL import Image

EXPECTED_COUNTS = {
    "L1": 5714,
    "L2": 4958,
    "L3": 4579,
    "L4": 4296,
    "L5": 4033,
    "L6": 3870,
    "L7": 2706,
    "L8": 3173,
    "L9": 2971,
}

validation_rows = []

for experiment, expected in EXPECTED_COUNTS.items():
    experiment_dir = DATASET_ROOT / experiment
    image_paths = sorted(
        experiment_dir.glob("*.png"),
        key=lambda path: int(path.stem),
    )

    found = len(image_paths)

    if image_paths:
        with Image.open(image_paths[0]) as image:
            image_size = image.size
            image_mode = image.mode
        first_frame = int(image_paths[0].stem)
        last_frame = int(image_paths[-1].stem)
    else:
        image_size = None
        image_mode = None
        first_frame = None
        last_frame = None

    validation_rows.append({
        "experimento": experiment,
        "esperado": expected,
        "encontrado": found,
        "primeiro_frame": first_frame,
        "ultimo_frame": last_frame,
        "dimensoes": str(image_size),
        "modo": image_mode,
        "status": (
            "OK"
            if found == expected and image_size == (400, 400)
            else "VERIFICAR"
        ),
    })

validation_df = pd.DataFrame(validation_rows)
display(validation_df)

total_found = int(validation_df["encontrado"].sum())
all_ok = bool((validation_df["status"] == "OK").all())

print(f"Total de imagens: {total_found:,}")
print(f"Validação concluída: {all_ok}")

if not all_ok:
    raise ValueError("Há divergências no conjunto de imagens.")

### 3. Construção da tabela-base de metadados

A tabela `metadados_df` funciona como um índice determinístico do conjunto de dados. Cada linha associa uma imagem ao respectivo experimento, ao número do *frame* e ao caminho relativo do arquivo. A ordenação numérica dos *frames* cria uma referência comum para localizar cada imagem; a metodologia `_old` terá sua ordem específica registrada separadamente, enquanto a metodologia `_new` utilizará esta sequência temporal. O caminho relativo é armazenado, em vez do caminho temporário completo, para que a tabela continue reutilizável em novas sessões do Colab.

In [ ]:
METADADOS_PATH = OUTPUT_DIR / "metadados_imagens_base.csv"
metadados_rows = []

for experiment in EXPECTED_COUNTS:
    experiment_dir = DATASET_ROOT / experiment
    image_paths = sorted(
        experiment_dir.glob("*.png"),
        key=lambda path: int(path.stem),
    )

    for image_path in image_paths:
        metadados_rows.append({
            "experimento": experiment,
            "frame": int(image_path.stem),
            "caminho_relativo": image_path.relative_to(DATASET_ROOT).as_posix(),
        })

metadados_df = (
    pd.DataFrame(metadados_rows)
    .sort_values(["experimento", "frame"])
    .reset_index(drop=True)
)
metadados_df.insert(
    0,
    "indice_global",
    range(len(metadados_df)),
)

resumo_metadados_df = (
    metadados_df
    .groupby("experimento", as_index=False)
    .agg(
        contagem=("frame", "size"),
        primeiro_frame=("frame", "min"),
        ultimo_frame=("frame", "max"),
    )
)
resumo_metadados_df["sequencia_continua"] = (
    (resumo_metadados_df["primeiro_frame"] == 0)
    & (
        resumo_metadados_df["contagem"]
        == resumo_metadados_df["ultimo_frame"]
        - resumo_metadados_df["primeiro_frame"]
        + 1
    )
)

total_esperado = sum(EXPECTED_COUNTS.values())
possui_duplicados = bool(
    metadados_df.duplicated(["experimento", "frame"]).any()
)
contagens_corretas = (
    resumo_metadados_df.set_index("experimento")["contagem"].to_dict()
    == EXPECTED_COUNTS
)
sequencias_completas = bool(
    resumo_metadados_df["sequencia_continua"].all()
)

if len(metadados_df) != total_esperado:
    raise ValueError(
        f"A tabela de metadados possui {len(metadados_df)} imagens; "
        f"eram esperadas {total_esperado}."
    )
if possui_duplicados:
    raise ValueError("Foram encontrados frames duplicados nos metadados.")
if not contagens_corretas or not sequencias_completas:
    raise ValueError(
        "A sequência de referência possui lacunas ou contagens divergentes."
    )

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
metadados_df.to_csv(METADADOS_PATH, index=False)

display(resumo_metadados_df)
display(metadados_df.head())

print(f"Linhas da tabela-base de metadados: {len(metadados_df):,}")
print(f"Frames duplicados: {possui_duplicados}")
print(f"Metadados salvos em: {METADADOS_PATH}")

### 4. Metodologia `_old` — reprodução da ordem e dos recortes por índice

Na metodologia `_old`, os arquivos de cada experimento foram obtidos com `os.listdir()`, sem ordenação numérica. A função de extração também iniciou o processamento na posição 30, fazendo com que o primeiro lote de 30 imagens não fosse utilizado. Somente depois da extração das *features* foram aplicados os intervalos com `iloc[início:fim]` em cada experimento. Portanto, esses limites representam posições no DataFrame de *features*, e não os números dos *frames*, pois a ordem dos *frames* não acompanha a ordem das *features*. Esse comportamento será reproduzido para manter a compatibilidade com o código de referência. Como a ordem retornada por `os.listdir()` pode variar entre sistemas de arquivos, ela será registrada nos metadados `_old`. Na metodologia `_new`, utilizaremos a ordenação numérica e um procedimento determinístico.

Na metodologia `_old`, o código usa `os.listdir()` sem ordenar os arquivos. Portanto, a posição de uma imagem na lista não corresponde necessariamente ao número do seu *frame*. Além disso, o primeiro lote de 30 imagens é ignorado. Depois disso, intervalos como `iloc[1450:5713]` selecionam posições no DataFrame de *features*, e não os arquivos `1450.png` até `5712.png`.

No L1, havia 5.714 imagens. Após o descarte de 30, restaram 5.684 linhas; como o recorte começou na posição 1450, foram selecionadas 4.234 imagens. Essa ordem foi registrada para reproduzir a metodologia `_old`. Na metodologia `_new`, os arquivos serão ordenados numericamente para preservar a sequência temporal.

In [ ]:
import os

IMAGENS_DESCARTADAS_OLD = 30

INTERVALOS_INDICES_OLD = {
    "L1": (1450, 5713),
    "L2": (600, 4550),
    "L3": (350, 4200),
    "L4": (750, 3800),
    "L5": (600, 3700),
    "L6": (650, 3600),
    "L7": (650, 2400),
    "L8": (800, 2650),
    "L9": (750, 2650),
}

ORDEM_OLD_PATH = OUTPUT_DIR / "metadados_ordem_old.csv"
RECORTES_OLD_PATH = OUTPUT_DIR / "metadados_recortes_old.csv"

indice_global_lookup = (
    metadados_df
    .set_index(["experimento", "frame"])["indice_global"]
    .to_dict()
)

ordem_old_partes = []  # A ordem completa de cada experimento.
recortes_old_partes = []  # O recorte de cada experimento.
resumo_recortes_old = []  # As informações resumidas de cada recorte.

for experimento, (inicio, fim) in INTERVALOS_INDICES_OLD.items():
    pasta_experimento = DATASET_ROOT / experimento

    # Reprodução deliberada do os.listdir() sem ordenação da metodologia _old.
    ##########################################################
    nomes_na_ordem_old = os.listdir(pasta_experimento)
    ##########################################################
    # Essa parte procura arquivos cuja extensão não seja .png.
    arquivos_nao_png = [
        nome
        for nome in nomes_na_ordem_old
        if Path(nome).suffix.lower() != ".png"
    ]
    if arquivos_nao_png:
        raise ValueError(
            f"Foram encontrados arquivos não PNG em {experimento}: "
            f"{arquivos_nao_png[:5]}"
        )

    # A metodologia _old começa em image_paths[30:].
    nomes_processados_old = nomes_na_ordem_old[IMAGENS_DESCARTADAS_OLD:]

    linhas_experimento_old = []
    for posicao_extracao_old, nome_arquivo in enumerate(nomes_processados_old):
        frame = int(Path(nome_arquivo).stem)
        linhas_experimento_old.append({
            "indice_global": indice_global_lookup[(experimento, frame)],
            "experimento": experimento,
            "frame": frame,
            "posicao_extracao_old": posicao_extracao_old,
            "caminho_relativo": f"{experimento}/{nome_arquivo}",
        })

    # As linhas são convertidas em um DataFrame. Em seguida, esse DataFrame é
    # guardado na lista ordem_old_partes.
    ordem_experimento_old_df = pd.DataFrame(linhas_experimento_old)
    ordem_old_partes.append(ordem_experimento_old_df)

    # Equivale ao iloc[inicio:fim] aplicado após a extração das features _old.
    recorte_experimento_old_df = ordem_experimento_old_df.iloc[inicio:fim].copy() ###??????? sera que nao eh isso que ta ferrando a ordem nao

    # A coluna seguinte registra a posição da imagem dentro desse novo recorte.
    # Ela começa novamente em zero: 0, 1, 2, 3, ...
    recorte_experimento_old_df["posicao_no_recorte_old"] = range(
        len(recorte_experimento_old_df)
    )
    recortes_old_partes.append(recorte_experimento_old_df)

    # Esse cálculo considera que o limite final pode ultrapassar o tamanho do DataFrame.
    quantidade_esperada_old = max(
        0,
        min(fim, len(ordem_experimento_old_df))
        - min(inicio, len(ordem_experimento_old_df)),
    )

    resumo_recortes_old.append({
        "experimento": experimento,
        "arquivos_na_pasta": len(nomes_na_ordem_old),
        "descartados_no_inicio_old": IMAGENS_DESCARTADAS_OLD,
        "linhas_apos_descarte_old": len(ordem_experimento_old_df),
        "indice_inicial_old": inicio,
        "indice_final_exclusivo_old": fim,
        "linhas_selecionadas_old": len(recorte_experimento_old_df),
        "limite_final_disponivel": fim <= len(ordem_experimento_old_df),
        "status": (
            "OK"
            if len(recorte_experimento_old_df) == quantidade_esperada_old
            else "VERIFICAR"
        ),
    })

# União dos experimentos: une a ordem completa e os recortes de L1 a L9.
# O ignore_index=True cria uma nova sequência de índices para o resultado completo.
metadados_ordem_old_df = pd.concat(
    ordem_old_partes,
    ignore_index=True,
)
metadados_recortes_old_df = pd.concat(
    recortes_old_partes,
    ignore_index=True,
)

# Identifica cada linha dentro do conjunto final da metodologia _old.
metadados_recortes_old_df.insert(
    0,
    "indice_old",
    range(len(metadados_recortes_old_df)),
)
resumo_recortes_old_df = pd.DataFrame(resumo_recortes_old)

# Validações finais.
if not (resumo_recortes_old_df["status"] == "OK").all():
    raise ValueError("Os recortes _old não correspondem aos limites definidos.")
if metadados_recortes_old_df["indice_global"].duplicated().any():
    raise ValueError("Uma mesma imagem apareceu mais de uma vez nos recortes _old.")

# Salva os arquivos .csv.
metadados_ordem_old_df.to_csv(ORDEM_OLD_PATH, index=False)
metadados_recortes_old_df.to_csv(RECORTES_OLD_PATH, index=False)

display(resumo_recortes_old_df)
display(metadados_recortes_old_df.head())

print(
    "Imagens _old após o descarte das 30 primeiras posições: "
    f"{len(metadados_ordem_old_df):,}"
)
print(
    "Imagens _old após os recortes por índice: "
    f"{len(metadados_recortes_old_df):,}"
)
print(f"Ordem _old registrada em: {ORDEM_OLD_PATH}")
print(f"Recortes _old registrados em: {RECORTES_OLD_PATH}")

* `indice_old`: posição da imagem no conjunto produzido pela metodologia `_old`;
* `indice_global`: identificação da mesma imagem na tabela-base;
* `experimento`: experimento de L1 a L9;
* `frame`: número original do arquivo;
* `posicao_extracao_old`: posição da imagem após o descarte inicial e antes do recorte;
* `caminho_relativo`: caminho utilizado para carregar a imagem;
* `posicao_no_recorte_old`: posição da imagem dentro do recorte daquele experimento.

### 5. Configuração da InceptionResNetV2

Nesta etapa, a rede InceptionResNetV2 é carregada com pesos pré-treinados no ImageNet, remoção da camada classificadora com `include_top=False` e aplicação de *global max pooling* com `pooling="max"`. Assim, a rede será utilizada somente como extratora e produzirá 1.536 *features* para cada imagem. Essa configuração é comum às duas metodologias; as diferenças estarão na seleção dos dados e nas etapas posteriores. Primeiro, a configuração será validada com uma única imagem antes do processamento completo do conjunto.

In [ ]:
import numpy as np
import tensorflow as tf
from tensorflow import keras

SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)

dispositivos_gpu = tf.config.list_physical_devices("GPU")

extrator_inceptionresnetv2 = keras.applications.InceptionResNetV2(
    weights="imagenet",
    include_top=False,
    pooling="max",
)
extrator_inceptionresnetv2.trainable = False

print(f"TensorFlow: {tf.__version__}")
print(f"GPU disponível: {bool(dispositivos_gpu)}")
print(f"Formato de entrada: {extrator_inceptionresnetv2.input_shape}")
print(f"Formato de saída: {extrator_inceptionresnetv2.output_shape}")

#### Teste da extração com uma imagem

A primeira imagem dos metadados `_old` será carregada em seu tamanho original de 400 × 400 pixels e convertida em um lote com uma única amostra. Em seguida, será aplicado o pré-processamento específico da InceptionResNetV2, que transforma os valores dos pixels para o intervalo esperado pela rede. A saída deverá possuir o formato `(1, 1536)`: uma imagem representada por 1.536 *features*.

In [ ]:
linha_teste = metadados_recortes_old_df.iloc[0]
caminho_imagem_teste = (
    DATASET_ROOT / linha_teste["caminho_relativo"]
)

imagem_teste = keras.utils.load_img(caminho_imagem_teste)
imagem_array = keras.utils.img_to_array(imagem_teste)
imagem_batch = np.expand_dims(imagem_array, axis=0)
#processar imagem
#(30, 400, 400, 3) (30 imagens, com 400 pixeis de largura, 400 de altura, e 3 RGBs ) - (quantidade de imagens, altura, largura, canais)

#normalizar os valores de RGB:
#Pixel 0     → -1
#Pixel 127,5 →  0
#Pixel 255   →  1

imagem_preprocessada = (
    keras.applications.inception_resnet_v2.preprocess_input(
        imagem_batch.copy()
    )
)


features_teste = extrator_inceptionresnetv2.predict(
    imagem_preprocessada,
    verbose=0,
)

print(f"Experimento: {linha_teste['experimento']}")
print(f"Frame: {linha_teste['frame']}")
print(f"Imagem: {caminho_imagem_teste}")
print(f"Formato da imagem: {imagem_array.shape}")
print(f"Formato do lote: {imagem_batch.shape}")
print(
    "Intervalo após o pré-processamento: "
    f"[{imagem_preprocessada.min():.3f}, "
    f"{imagem_preprocessada.max():.3f}]"
)
print(f"Formato das features: {features_teste.shape}")

if features_teste.shape != (1, 1536):
    raise ValueError(
        f"Formato inesperado das features: {features_teste.shape}"
    )

### 6. Metodologia `_old` — extração das *features* por experimento

Nesta etapa, as imagens selecionadas pelos recortes da metodologia `_old` serão processadas pela InceptionResNetV2 em lotes de 30. Para reduzir o risco de perder todo o processamento caso a sessão do Colab seja interrompida, as *features* serão salvas separadamente para cada experimento, de L1 a L9. Cada arquivo terá uma linha por imagem e 1.536 colunas, mantendo exatamente a ordem de `metadados_recortes_old_df`.

Antes de processar um experimento, o código verificará se já existe um arquivo com as dimensões esperadas. Nesse caso, ele será reutilizado. Esta etapa gera apenas as representações visuais; os valores de largura e as variáveis de processo serão associados posteriormente.

In [ ]:
import gc
from time import perf_counter
from tqdm.auto import tqdm

BATCH_SIZE = 30
FEATURES_DIR_OLD = OUTPUT_DIR / "features_old"
RESUMO_FEATURES_OLD_PATH = OUTPUT_DIR / "resumo_features_old.csv"
FEATURES_DIR_OLD.mkdir(parents=True, exist_ok=True)

if not tf.config.list_physical_devices("GPU"):
    raise RuntimeError(
        "A GPU não foi detectada. Ative uma GPU antes da extração completa."
    )

resumo_features_old = []

#experimentos
for experimento in EXPECTED_COUNTS:
    metadados_experimento_old_df = (
        metadados_recortes_old_df[
            metadados_recortes_old_df["experimento"] == experimento
        ]
        .reset_index(drop=True)
    )

    features_path_old = FEATURES_DIR_OLD / f"{experimento}_features_old.npy"
    formato_esperado = (len(metadados_experimento_old_df), 1536)
    inicio_experimento = perf_counter()

    # Se o experimento já tiver sido concluído, reutiliza o arquivo salvo.
    if features_path_old.exists():
        features_experimento_old = np.load(features_path_old)

        if features_experimento_old.shape != formato_esperado:
            raise ValueError(
                f"{features_path_old.name} possui formato "
                f"{features_experimento_old.shape}; esperado {formato_esperado}."
            )
        if not np.isfinite(features_experimento_old).all():
            raise ValueError(
                f"{features_path_old.name} contém valores NaN ou infinitos."
            )

        status = "reutilizado"

    else:
        features_lotes = []
        inicios_lotes = range(
            0,
            len(metadados_experimento_old_df),
            BATCH_SIZE,
        )
        #lotes do experimento
        for inicio_lote in tqdm(
            inicios_lotes,
            desc=f"Extraindo {experimento}",
        ):
            fim_lote = inicio_lote + BATCH_SIZE
            metadados_lote_df = metadados_experimento_old_df.iloc[
                inicio_lote:fim_lote
            ]

            imagens_lote = [] #armazena os arrays da imagem
            #imagens dos lotes
            for caminho_relativo in metadados_lote_df["caminho_relativo"]:
                caminho_imagem = DATASET_ROOT / caminho_relativo
                imagem = keras.utils.load_img(caminho_imagem)
                imagem_array = keras.utils.img_to_array(imagem)
                imagens_lote.append(imagem_array)

            # Empilha as imagens para formar um lote: (N, 400, 400, 3). e pre processa
            imagens_lote = np.stack(imagens_lote, axis=0)
            imagens_lote = (
                keras.applications.inception_resnet_v2.preprocess_input(
                    imagens_lote
                )
            )

            features_lote = extrator_inceptionresnetv2.predict(
                imagens_lote,
                verbose=0,
            )
            features_lotes.append(
                features_lote.astype(np.float32, copy=False)
            )

        # Une verticalmente os lotes do mesmo experimento.
        features_experimento_old = np.concatenate(
            features_lotes,
            axis=0,
        )

        if features_experimento_old.shape != formato_esperado:
            raise ValueError(
                f"As features de {experimento} possuem formato "
                f"{features_experimento_old.shape}; esperado {formato_esperado}."
            )
        if not np.isfinite(features_experimento_old).all():
            raise ValueError(
                f"As features de {experimento} contêm valores NaN ou infinitos."
            )

        np.save(features_path_old, features_experimento_old)
        status = "extraído"

    tempo_segundos = perf_counter() - inicio_experimento
    resumo_features_old.append({
        "experimento": experimento,
        "imagens": features_experimento_old.shape[0],
        "features_por_imagem": features_experimento_old.shape[1],
        "status": status,
        "tempo_segundos": tempo_segundos,
        "arquivo": str(features_path_old),
    })

    print(
        f"{experimento}: {features_experimento_old.shape} | "
        f"{status} | {tempo_segundos:.1f} s"
    )

    del features_experimento_old
    if "features_lotes" in locals():
        del features_lotes
    gc.collect()

resumo_features_old_df = pd.DataFrame(resumo_features_old)
resumo_features_old_df.to_csv(RESUMO_FEATURES_OLD_PATH, index=False)
display(resumo_features_old_df)

print(f"Resumo salvo em: {RESUMO_FEATURES_OLD_PATH}")

## 7. Consolidação e validação das *features* `_old`

Nesta etapa, os nove arquivos de *features* serão carregados na ordem dos experimentos L1–L9 e comparados com `metadados_recortes_old_df`. O código verificará a quantidade de linhas, as 1.536 colunas e a ausência de valores inválidos em cada arquivo. Em seguida, as matrizes serão unidas verticalmente e salvas em um único arquivo, sem alterar a ordem das imagens. Os arquivos separados serão mantidos, pois permitem recuperar somente um experimento quando necessário.

In [ ]:
FEATURES_CONSOLIDADAS_OLD_PATH = (
    OUTPUT_DIR / "features_inceptionresnetv2_old.npy"
)

features_partes_old = []
resumo_consolidacao_old = []
linha_inicial = 0

for experimento in EXPECTED_COUNTS:
    # Seleciona os metadados do experimento na mesma ordem usada na extração.
    metadados_experimento_old_df = (
        metadados_recortes_old_df[
            metadados_recortes_old_df["experimento"] == experimento
        ]
        .reset_index(drop=True)
    )

    features_path_old = FEATURES_DIR_OLD / f"{experimento}_features_old.npy"

    if not features_path_old.exists():
        raise FileNotFoundError(
            f"Arquivo de features não encontrado: {features_path_old}"
        )

    # mmap_mode='r' permite validar o arquivo sem copiá-lo inteiro para a memória.
    features_experimento_old = np.load(features_path_old, mmap_mode="r")
    formato_esperado = (len(metadados_experimento_old_df), 1536)

    if features_experimento_old.shape != formato_esperado:
        raise ValueError(
            f"{features_path_old.name} possui formato "
            f"{features_experimento_old.shape}; esperado {formato_esperado}."
        )

    if not np.isfinite(features_experimento_old).all():
        raise ValueError(
            f"{features_path_old.name} contém valores NaN ou infinitos."
        )

    linha_final_exclusiva = linha_inicial + len(features_experimento_old)

    features_partes_old.append(features_experimento_old)
    resumo_consolidacao_old.append({
        "experimento": experimento,
        "linhas_metadados": len(metadados_experimento_old_df),
        "linhas_features": features_experimento_old.shape[0],
        "features_por_imagem": features_experimento_old.shape[1],
        "linha_inicial": linha_inicial,
        "linha_final_exclusiva": linha_final_exclusiva,
        "status": "OK",
    })

    linha_inicial = linha_final_exclusiva

# Une L1–L9 verticalmente, preservando a ordem de metadados_recortes_old_df.
features_consolidadas_old = np.concatenate(
    features_partes_old,
    axis=0,
).astype(np.float32, copy=False)

formato_esperado_consolidado = (
    len(metadados_recortes_old_df),
    1536,
)

if features_consolidadas_old.shape != formato_esperado_consolidado:
    raise ValueError(
        "A matriz consolidada possui formato "
        f"{features_consolidadas_old.shape}; "
        f"esperado {formato_esperado_consolidado}."
    )

if not np.isfinite(features_consolidadas_old).all():
    raise ValueError(
        "A matriz consolidada contém valores NaN ou infinitos."
    )

np.save(
    FEATURES_CONSOLIDADAS_OLD_PATH,
    features_consolidadas_old,
)

resumo_consolidacao_old_df = pd.DataFrame(resumo_consolidacao_old)
display(resumo_consolidacao_old_df)

print(f"Formato consolidado: {features_consolidadas_old.shape}")
print(f"Tipo dos dados: {features_consolidadas_old.dtype}")
print(
    "Tamanho em memória: "
    f"{features_consolidadas_old.nbytes / (1024 ** 2):.1f} MB"
)
print(f"Arquivo salvo em: {FEATURES_CONSOLIDADAS_OLD_PATH}")

## 8. Inclusão das larguras e dos parâmetros de processo — metodologia `_old`

As larguras do cordão foram medidas em pontos discretos de cada experimento. Para gerar um valor de largura `y` para cada imagem selecionada, a metodologia `_old` ajusta uma spline cúbica natural aos valores medidos e a avalia em posições igualmente espaçadas no intervalo de 0 a 8. A associação é feita pela posição da linha dentro do recorte de cada experimento, reproduzindo o código original, e não pelo número do *frame*. Nesta etapa também são incluídos os valores constantes de corrente, velocidade do bico e vazão mássica definidos para L1–L9. As *features* permanecem no arquivo `.npy`; este código salva em CSV apenas os metadados, os parâmetros e o alvo `y`, evitando duplicar a matriz de 1.536 colunas.

In [ ]:
from scipy.interpolate import CubicSpline

# Larguras medidas experimentalmente ao longo de cada cordão.
LARGURAS_MEDIDAS = {
    "L1": np.array([
        8.270, 8.737, 8.775, 8.911, 8.873, 9.184, 9.741, 9.776,
        9.552, 9.477, 9.696, 9.614, 10.184, 9.960, 9.918, 10.024,
        10.096, 10.430, 10.506, 10.094, 10.319, 10.504, 10.542,
        10.728, 10.954, 10.729, 10.767, 11.040, 11.148, 11.108,
        10.696, 11.072, 11.477, 11.737, 11.811, 11.591,
    ]),
    "L2": np.array([
        10.179, 10.655, 10.568, 10.950, 11.422, 12.185, 12.314,
        12.440, 12.606, 12.650, 12.263, 12.525, 13.371, 12.986,
        12.598, 12.896, 13.108, 12.940, 12.939, 12.938, 13.278,
        13.153, 12.941, 13.026, 12.855, 12.604, 12.556, 12.981,
        13.066, 12.726, 13.150, 13.448, 13.490, 13.405, 13.022,
        12.638,
    ]),
    "L3": np.array([
        14.613, 14.794, 15.011, 14.781, 14.743, 14.581, 14.305,
        14.575, 15.756, 14.845, 14.726, 15.030, 15.331, 15.444,
        15.330, 15.027, 15.406, 15.559, 15.294, 14.910, 15.363,
        14.946, 15.555, 16.389, 16.807, 16.997, 16.160, 15.133,
        14.905, 14.411,
    ]),
    "L4": np.array([
        5.969, 7.203, 7.412, 7.881, 8.045, 8.256, 8.179, 8.132,
        8.182, 8.306, 8.221, 8.303, 8.596, 8.638, 8.763, 8.552,
        8.634, 8.845, 8.763, 8.805, 9.093, 9.053, 8.839, 9.262,
        9.306, 9.432, 9.305, 9.644, 9.433, 9.096, 8.927, 9.183,
        9.361, 9.736, 9.902, 9.899, 10.027, 9.982, 9.982, 9.898,
        9.982, 10.406, 10.234, 10.192, 10.234, 10.191, 9.811,
    ]),
    "L5": np.array([
        9.947, 10.231, 10.594, 10.838, 11.003, 11.083, 11.204,
        11.286, 11.285, 11.202, 11.488, 11.366, 11.446, 11.366,
        11.367, 11.406, 11.528, 11.650, 11.487, 11.202, 11.202,
        11.325, 11.853, 11.690, 11.568, 11.650, 11.407, 11.162,
        11.204, 11.449, 11.816, 11.775, 11.815, 11.732, 11.570,
        11.488, 11.080, 11.242, 12.139, 12.587, 13.238, 12.872,
        12.383,
    ]),
    "L6": np.array([
        11.711, 11.688, 11.530, 11.597, 11.733, 11.802, 12.050,
        12.413, 12.367, 12.713, 12.553, 12.484, 12.461, 12.527,
        12.458, 12.100, 12.898, 12.981, 13.479, 12.754, 12.866,
        12.935, 12.980, 13.772, 13.684, 13.524, 13.595, 13.078,
        13.395, 13.144, 13.004, 12.889, 12.416, 12.099,
    ]),
    "L7": np.array([
        6.370, 6.371, 6.158, 6.328, 6.369, 6.709, 6.752, 6.795,
        6.711, 6.626, 6.584, 6.413, 6.455, 6.328, 6.539, 6.497,
        6.667, 6.539, 6.497, 6.709, 6.497, 6.631, 6.795, 6.880,
        7.176, 7.304, 7.559, 7.898, 8.113, 8.113, 8.241, 8.282,
        8.324, 8.198, 8.028, 7.857, 7.858, 7.772, 7.559,
    ]),
    "L8": np.array([
        7.545, 8.719, 8.588, 8.373, 8.853, 9.148, 9.391, 9.258,
        9.071, 9.443, 8.746, 8.904, 8.877, 8.904, 8.984, 8.717,
        9.198, 8.476, 8.048, 8.342, 9.332, 9.199, 8.587, 8.986,
        9.466, 9.467, 9.146, 8.931, 9.039, 9.466, 9.733, 9.947,
        10.054, 10.187, 9.920, 9.522,
    ]),
    "L9": np.array([
        8.070, 8.383, 8.756, 9.814, 10.610, 11.141, 10.769,
        10.186, 9.761, 9.974, 10.186, 10.611, 10.292, 10.241,
        10.399, 10.613, 10.191, 9.979, 10.139, 9.873, 9.923,
        9.923, 9.553, 9.765, 9.818, 9.392, 9.498, 9.607, 9.768,
        9.447, 10.089, 10.245, 10.354, 10.197, 10.354, 9.826,
        9.779, 9.514,
    ]),
}

# Parâmetros constantes definidos para cada experimento do planejamento Taguchi.
PARAMETROS_PROCESSO = {
    "L1": {"current": 120, "nozzle_speed": 60, "mass_flow_rate": 10},
    "L2": {"current": 160, "nozzle_speed": 60, "mass_flow_rate": 15},
    "L3": {"current": 200, "nozzle_speed": 60, "mass_flow_rate": 20},
    "L4": {"current": 120, "nozzle_speed": 80, "mass_flow_rate": 10},
    "L5": {"current": 160, "nozzle_speed": 80, "mass_flow_rate": 15},
    "L6": {"current": 200, "nozzle_speed": 80, "mass_flow_rate": 20},
    "L7": {"current": 120, "nozzle_speed": 100, "mass_flow_rate": 10},
    "L8": {"current": 160, "nozzle_speed": 100, "mass_flow_rate": 15},
    "L9": {"current": 200, "nozzle_speed": 100, "mass_flow_rate": 20},
}

def criar_spline_largura(experimento):
    larguras = LARGURAS_MEDIDAS[experimento]
    posicoes_medidas = np.arange(0, 8, 8 / len(larguras))
    if len(posicoes_medidas) != len(larguras):
        raise ValueError(f"Quantidade de posições incompatível com {experimento}.")
    return CubicSpline(posicoes_medidas, larguras, bc_type="natural")

METADADOS_ROTULADOS_OLD_PATH = OUTPUT_DIR / "metadados_rotulados_old.csv"
metadados_rotulados_old_partes = []
resumo_rotulagem_old = []

for experimento in EXPECTED_COUNTS:
    dados_experimento_old_df = (
        metadados_recortes_old_df[
            metadados_recortes_old_df["experimento"] == experimento
        ]
        .copy()
        .reset_index(drop=True)
    )

    posicoes_interpolacao_old = np.arange(
        0, 8, 8 / len(dados_experimento_old_df)
    )
    if len(posicoes_interpolacao_old) != len(dados_experimento_old_df):
        raise ValueError(
            f"A interpolação de {experimento} gerou "
            f"{len(posicoes_interpolacao_old)} posições para "
            f"{len(dados_experimento_old_df)} imagens."
        )

    spline_largura = criar_spline_largura(experimento)
    parametros = PARAMETROS_PROCESSO[experimento]

    dados_experimento_old_df["posicao_interpolacao_old"] = posicoes_interpolacao_old
    dados_experimento_old_df["current"] = parametros["current"]
    dados_experimento_old_df["nozzle_speed"] = parametros["nozzle_speed"]
    dados_experimento_old_df["mass_flow_rate"] = parametros["mass_flow_rate"]
    dados_experimento_old_df["y"] = spline_largura(posicoes_interpolacao_old)

    metadados_rotulados_old_partes.append(dados_experimento_old_df)
    resumo_rotulagem_old.append({
        "experimento": experimento,
        "larguras_medidas": len(LARGURAS_MEDIDAS[experimento]),
        "imagens_rotuladas": len(dados_experimento_old_df),
        "y_inicial": dados_experimento_old_df["y"].iloc[0],
        "y_final": dados_experimento_old_df["y"].iloc[-1],
        "y_minimo": dados_experimento_old_df["y"].min(),
        "y_maximo": dados_experimento_old_df["y"].max(),
    })

metadados_rotulados_old_df = pd.concat(
    metadados_rotulados_old_partes, ignore_index=True
)
resumo_rotulagem_old_df = pd.DataFrame(resumo_rotulagem_old)

if len(metadados_rotulados_old_df) != features_consolidadas_old.shape[0]:
    raise ValueError("A quantidade de rótulos não corresponde à quantidade de features.")
if not np.isfinite(metadados_rotulados_old_df["y"]).all():
    raise ValueError("Foram encontrados valores inválidos na coluna y.")
if not np.array_equal(
    metadados_rotulados_old_df["indice_old"].to_numpy(),
    np.arange(len(metadados_rotulados_old_df)),
):
    raise ValueError("A ordem dos metadados _old foi alterada.")

metadados_rotulados_old_df.to_csv(METADADOS_ROTULADOS_OLD_PATH, index=False)

display(resumo_rotulagem_old_df)
display(
    metadados_rotulados_old_df[
        [
            "indice_old",
            "experimento",
            "frame",
            "posicao_interpolacao_old",
            "current",
            "nozzle_speed",
            "mass_flow_rate",
            "y",
        ]
    ].head()
)

print(f"Linhas rotuladas: {len(metadados_rotulados_old_df):,}")
print(f"Linhas rotuladas: {len(metadados_rotulados_old_df):,}")
print(f"Metadados rotulados salvos em: {METADADOS_ROTULADOS_OLD_PATH}")

**Na metodologia _old, as imagens são obtidas com `os.listdir()` sem ordenação numérica. Assim, os valores interpolados de y são associados à posição das imagens na lista, e não à sequência real dos frames, comprometendo a correspondência temporal entre imagem e largura. Esse comportamento será mantido apenas para reprodução e corrigido na metodologia `_new`.**

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D

# Curvas e pontos originais no mesmo eixo; uma cor por experimento.
with plt.rc_context({'font.size': 12, 'axes.titlesize': 16, 'axes.labelsize': 13,
                     'legend.fontsize': 11, 'font.weight': 'normal'}):
    figura, eixo = plt.subplots(figsize=(15, 8), layout='constrained')
    cores = plt.get_cmap('tab10')
    for numero, experimento in enumerate(EXPECTED_COUNTS):
        cor = cores(numero % 10)
        larguras = LARGURAS_MEDIDAS[experimento]
        posicoes_medidas = np.arange(0, 8, 8 / len(larguras))
        dados = metadados_rotulados_old_df.loc[
            metadados_rotulados_old_df['experimento'] == experimento
        ].sort_values('posicao_interpolacao_old')
        eixo.plot(dados['posicao_interpolacao_old'], dados['y'],
                  color=cor, linewidth=1.8, label=experimento)
        eixo.scatter(posicoes_medidas, larguras, color=cor, s=32,
                     edgecolors='white', linewidths=0.5, zorder=3)
    eixo.set(title='Interpolação das larguras — todos os experimentos (metodologia _old)',
             xlabel='Posição ao longo do cordão', ylabel='Largura (mm)')
    eixo.grid(alpha=0.25)
    eixo.set_axisbelow(True)
    eixo.margins(x=0.02, y=0.06)
    handles, labels = eixo.get_legend_handles_labels()
    handles.append(Line2D([], [], color='#555555', marker='o', linestyle='None',
                          markersize=6, label='Pontos usados na interpolação'))
    eixo.legend(handles=handles, loc='upper left', bbox_to_anchor=(1.02, 1),
                borderaxespad=0, frameon=False, title='Experimentos')
    plt.show()


## 9. Construção do dataset estático — metodologia `_old`

Nesta etapa, a matriz consolidada de 1.536 *features* será associada aos parâmetros de processo, ao identificador do experimento e ao valor de largura `y`. A ordem das linhas será preservada para reproduzir o arquivo estático utilizado por Fernandes Balbino. O resultado terá 26.634 linhas e 1.541 colunas e será salvo em CSV para utilização na etapa de geração das variáveis temporais.

In [ ]:
STATIC_DATASET_OLD_PATH = (
    OUTPUT_DIR / "static_inceptionresnetv2_merged_df_old.csv"
)

# As colunas 0–1535 correspondem às features da InceptionResNetV2.
colunas_features = list(range(features_consolidadas_old.shape[1]))
static_df_old = pd.DataFrame(
    features_consolidadas_old,
    columns=colunas_features,
)

# Mantém a ordem original esperada pelo código de referência.
colunas_adicionais = [
    "current",
    "nozzle_speed",
    "mass_flow_rate",
    "experimento",
    "y",
]
for coluna in colunas_adicionais:
    static_df_old[coluna] = metadados_rotulados_old_df[coluna].to_numpy()

formato_esperado_static_old = (
    len(metadados_rotulados_old_df),
    features_consolidadas_old.shape[1] + len(colunas_adicionais),
)

if static_df_old.shape != formato_esperado_static_old:
    raise ValueError(
        f"O dataset estático possui formato {static_df_old.shape}; "
        f"esperado {formato_esperado_static_old}."
    )
if not np.isfinite(
    static_df_old[colunas_features].to_numpy()
).all():
    raise ValueError("O dataset contém features inválidas.")
if not np.isfinite(static_df_old["y"]).all():
    raise ValueError("O dataset contém valores inválidos em y.")

static_df_old.to_csv(STATIC_DATASET_OLD_PATH, index=False)

# Exibe somente algumas features e as colunas adicionais.
colunas_visualizacao = colunas_features[:5] + colunas_adicionais
display(static_df_old[colunas_visualizacao].head())

tamanho_csv_mb = STATIC_DATASET_OLD_PATH.stat().st_size / (1024 ** 2)
print(f"Formato do dataset estático: {static_df_old.shape}")
print(f"Tamanho do CSV: {tamanho_csv_mb:.1f} MB")
print(f"Dataset salvo em: {STATIC_DATASET_OLD_PATH}")

## 10. Regressão estática — metodologia `_old`

Nesta etapa, treinaremos o modelo de referência **sem informação temporal**: as 1.536 features de cada imagem serão usadas para estimar a largura `y`. Os parâmetros de processo e o identificador do experimento permanecerão no dataset, mas não serão entradas da rede. Nenhuma imagem será processada novamente.

A configuração reproduz o modelo da primeira iteração (`i = 1`) da seção “Generation of 100 previous images using the most naive method” do [notebook de Fernandes Balbino](https://github.com/LeoFernanndes/temporal-cnn/blob/main/temporal_features_generation.ipynb):

- Divisão aleatória: 70% para treinamento e 30% para validação, com `random_state=42`.
- Rede: 1.536 entradas → camada densa com 32 neurônios e ReLU → uma saída linear.
- Treinamento: Adam, erro absoluto médio (MAE), até 300 épocas, lotes de 128 e parada antecipada após 20 épocas sem melhoria em `val_loss`, restaurando os melhores pesos.

Ao final, serão exibidos MAE em milímetros, MAPE em porcentagem e R². O modelo, as previsões, os índices da divisão, as métricas e o histórico serão salvos em uma pasta exclusiva para cada execução. `linha_dataset` identifica a linha original do dataset; não é o número do frame.

**Limitação da reprodução:** os mesmos 30% orientam a parada antecipada e fornecem as métricas; portanto, são dados de validação, não um teste independente. A ordem e a associação imagem–`y` da versão `_old` serão preservadas. Não será aplicada normalização, PCA ou qualquer correção do protocolo nesta etapa. A inicialização da rede e o ambiente podem causar variações nos resultados; `random_state=42` fixa apenas a divisão. O protocolo `_new` será desenvolvido em outro notebook.

**Como executar:** monte o Google Drive pela primeira célula do notebook e execute o código abaixo. Se `static_df_old` não estiver na memória, o CSV já salvo na etapa 9 será carregado automaticamente; não é necessário repetir a extração de features.


In [ ]:
from pathlib import Path
from datetime import datetime, timezone
from time import perf_counter
import numpy as np
import pandas as pd
from tensorflow import keras
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_absolute_percentage_error, r2_score

# Reutiliza o dataset da etapa 9; em uma nova sessão, carrega o CSV salvo.
# O Google Drive precisa estar montado, mas não é necessário extrair as imagens de novo.
OUTPUT_DIR = Path("/content/drive/MyDrive/TCC/resultados/inceptionresnetv2")
STATIC_DATASET_OLD_PATH = OUTPUT_DIR / "static_inceptionresnetv2_merged_df_old.csv"
if "static_df_old" not in globals():
    if not STATIC_DATASET_OLD_PATH.is_file():
        raise FileNotFoundError(
            "Monte o Drive e confira o CSV da etapa 9: "
            f"{STATIC_DATASET_OLD_PATH}"
        )
    static_df_old = pd.read_csv(STATIC_DATASET_OLD_PATH)

# Como no código de referência, parâmetros de processo e experimento NÃO entram em X.
# Não reordena as linhas, não recalcula y e não aplica normalização nem PCA.
colunas_excluidas = ["y", "current", "nozzle_speed", "mass_flow_rate", "experimento"]
X = static_df_old.drop(columns=colunas_excluidas).astype(np.float32)
y = static_df_old["y"]
if list(map(str, X.columns)) != [str(i) for i in range(1536)]:
    raise ValueError("X deve conter somente as features 0 a 1535, nessa ordem.")
if len(X) < 4 or not static_df_old.index.is_unique:
    raise ValueError("Confira a quantidade de linhas e o índice do dataset estático.")
if not np.isfinite(X.to_numpy()).all() or not np.isfinite(y.to_numpy()).all():
    raise ValueError("Foram encontrados valores NaN ou infinitos em X ou y.")

# Divisão aleatória da metodologia _old, misturando amostras de L1 a L9.
# random_state fixa a divisão; não fixa, por si só, a inicialização da rede.
X_train, X_val, y_train, y_val = train_test_split(
    X, y, train_size=0.7, random_state=42
)
print(f"Treino: {X_train.shape} | Validação: {X_val.shape}")

# Equivale ao modelo da primeira iteração (i=1), ainda sem features temporais.
modelo_estatico_old = keras.Sequential([
    keras.layers.Input(shape=(1536,)),
    keras.layers.Dense(32, activation="relu"),
    keras.layers.Dense(1),
])
early_stopping = keras.callbacks.EarlyStopping(
    monitor="val_loss",
    patience=20,
    verbose=0,
    mode="auto",
    baseline=None,
    restore_best_weights=True,
    start_from_epoch=0,
)
modelo_estatico_old.compile(optimizer="adam", loss="mean_absolute_error")
modelo_estatico_old.summary()

# Os mesmos 30% orientam o early stopping e as métricas finais, como na versão _old.
# Por isso, são chamados de validação: NÃO constituem um teste independente.
inicio_treino = perf_counter()
historico_estatico_old = modelo_estatico_old.fit(
    X_train, y_train,
    epochs=300,
    batch_size=128,
    validation_data=(X_val, y_val),
    callbacks=[early_stopping],
    verbose=2,
)
tempo_treino_s = perf_counter() - inicio_treino

# As previsões usam os melhores pesos restaurados pelo early stopping.
y_pred = modelo_estatico_old.predict(X_val, verbose=0).reshape(-1)
metricas_estaticas_old_df = pd.DataFrame([{
    "metodologia": "old",
    "particao": "validacao",
    "n_treino": len(y_train),
    "n_validacao": len(y_val),
    "MAE_mm": mean_absolute_error(y_val, y_pred),
    "MAPE_pct": 100 * mean_absolute_percentage_error(y_val, y_pred),
    "R2": r2_score(y_val, y_pred),
    "epocas_executadas": len(historico_estatico_old.history["loss"]),
    "melhor_epoca": int(np.argmin(historico_estatico_old.history["val_loss"]) + 1),
    "tempo_treino_s": tempo_treino_s,
}])

# Guarda a linha de origem para rastrear cada previsão (não é o número do frame).
previsoes_estaticas_old_df = pd.DataFrame({
    "linha_dataset": y_val.index.to_numpy(),
    "experimento": static_df_old.loc[y_val.index, "experimento"].to_numpy(),
    "y": y_val.to_numpy(),
    "y_pred": y_pred,
})
previsoes_estaticas_old_df["erro_absoluto_mm"] = np.abs(
    previsoes_estaticas_old_df["y"] - previsoes_estaticas_old_df["y_pred"]
)

# Uma pasta por execução evita sobrescrever resultados anteriores.
identificador_execucao = datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S_%f_UTC")
execucao_dir_old = OUTPUT_DIR / "regressao_estatica_old" / identificador_execucao
execucao_dir_old.mkdir(parents=True, exist_ok=False)
modelo_estatico_old.save(execucao_dir_old / "modelo_estatico_old.keras")
metricas_estaticas_old_df.to_csv(execucao_dir_old / "metricas_old.csv", index=False)
previsoes_estaticas_old_df.to_csv(execucao_dir_old / "previsoes_validacao_old.csv", index=False)
historico_df = pd.DataFrame(historico_estatico_old.history)
historico_df.insert(0, "epoca", np.arange(1, len(historico_df) + 1))
historico_df.to_csv(execucao_dir_old / "historico_old.csv", index=False)
pd.DataFrame({"linha_dataset": y_train.index}).to_csv(
    execucao_dir_old / "indices_treino_old.csv", index=False
)

display(metricas_estaticas_old_df)
display(previsoes_estaticas_old_df.head())
print(f"Resultados salvos em: {execucao_dir_old}")
print("Baseline estático _old concluído. Nenhuma feature temporal foi gerada nesta etapa.")


## 11. Primeira feature temporal — metodologia `_old`

Agora vamos reproduzir somente a primeira iteração temporal do código de referência. O modelo estático já treinado será usado para gerar uma previsão para cada linha de cada experimento. A nova coluna `1536` será formada por `[média de y do experimento, previsão da linha anterior, previsão da linha anterior, ...]`.

Nesta versão `_old`, “linha anterior” significa a linha anterior na ordem atual do dataset. Como a ordem foi preservada com `os.listdir()`, ela não representa necessariamente o frame cronológico anterior. A média é calculada usando todo o recorte do experimento, inclusive linhas que participaram da validação, preservando deliberadamente o procedimento original e sua limitação.

A célula também verifica se os experimentos estão em blocos contíguos, mantém todas as linhas e monta uma tabela de rastreamento. Nessa tabela, `linha_origem_previsao` mostra de onde veio o valor temporal; a primeira linha de cada experimento recebe a média e não uma previsão.

Neste momento, a rede **não é treinada novamente** com a nova coluna. Primeiro vamos verificar se o deslocamento ficou correto; depois essa entrada será usada para a iteração seguinte. O procedimento corrigido, sem essa forma de vazamento, será construído separadamente na metodologia `_new`.


In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import numpy as np
import pandas as pd
from tensorflow import keras

# Usa explicitamente a execução estática que acabamos de concluir.
# Em uma nova sessão, basta montar o Drive antes desta célula.
OUTPUT_DIR = Path("/content/drive/MyDrive/TCC/resultados/inceptionresnetv2")
EXECUCAO_ESTATICA_OLD_DIR = (
    OUTPUT_DIR / "regressao_estatica_old" / "20260920_174918_207096_UTC"
)
STATIC_DATASET_OLD_PATH = OUTPUT_DIR / "static_inceptionresnetv2_merged_df_old.csv"
MODELO_ESTATICO_OLD_PATH = EXECUCAO_ESTATICA_OLD_DIR / "modelo_estatico_old.keras"
for caminho in [STATIC_DATASET_OLD_PATH, MODELO_ESTATICO_OLD_PATH]:
    if not caminho.is_file():
        raise FileNotFoundError(f"Monte o Drive e confira o arquivo: {caminho}")

# Carrega os artefatos existentes; não extrai imagens nem treina outra rede.
static_df_old = pd.read_csv(STATIC_DATASET_OLD_PATH)
modelo_base_temporal_old = keras.models.load_model(
    MODELO_ESTATICO_OLD_PATH, compile=False
)
colunas_excluidas = ["y", "current", "nozzle_speed", "mass_flow_rate", "experimento"]
colunas_features = static_df_old.drop(columns=colunas_excluidas).columns.tolist()
if colunas_features != [str(j) for j in range(1536)]:
    raise ValueError("O CSV deve conter as features estáticas 0 a 1535, nessa ordem.")
if modelo_base_temporal_old.input_shape[-1] != 1536:
    raise ValueError("O modelo carregado não corresponde à regressão estática.")
if static_df_old["experimento"].isna().any() or static_df_old.empty:
    raise ValueError("Confira os identificadores dos experimentos no dataset.")

# A concatenação do código original pressupõe blocos contíguos por experimento.
# Apenas valida essa condição; não ordena ou modifica o dataset _old.
ordem_experimentos = static_df_old["experimento"].drop_duplicates().tolist()
ordem_blocos = static_df_old.loc[
    static_df_old["experimento"].ne(static_df_old["experimento"].shift()),
    "experimento",
].tolist()
if ordem_blocos != ordem_experimentos:
    raise ValueError("Os experimentos não estão em blocos contíguos como na etapa 9.")

# Primeira iteração do método _old: deslocamento de UMA LINHA, não de um frame garantido.
i = 1
nome_feature_temporal = str(1536 + i - 1)
dados_temporais_old = static_df_old.copy()
final_dislocated_predictions = []
tabelas_debug = []

for experimento in ordem_experimentos:
    dados_experimento = static_df_old[
        static_df_old["experimento"] == experimento
    ].copy()
    X_experimento = dados_experimento[colunas_features].to_numpy(dtype=np.float32)
    if not np.isfinite(X_experimento).all() or not np.isfinite(dados_experimento["y"]).all():
        raise ValueError(f"Dados inválidos no experimento {experimento}.")
    previsoes = modelo_base_temporal_old.predict(X_experimento, verbose=0).reshape(-1)
    if len(previsoes) != len(dados_experimento) or not np.isfinite(previsoes).all():
        raise ValueError(f"Previsões inválidas no experimento {experimento}.")

    # Reprodução deliberada: a média inclui TODO o y do experimento (treino e validação).
    # Essa informação do alvo será usada como entrada; a correção fica para a versão _new.
    media_y = dados_experimento["y"].mean()
    dislocated_ypred = [media_y] * i
    dislocated_ypred.extend(previsoes.tolist())
    dislocated_ypred = dislocated_ypred[:len(dados_experimento)]
    final_dislocated_predictions.extend(dislocated_ypred)

    # Mostra de qual linha veio a previsão; a primeira linha usa média, não previsão.
    indices_origem = pd.array(
        [pd.NA] * i + dados_experimento.index.tolist(), dtype="Int64"
    )[:len(dados_experimento)]
    tabelas_debug.append(pd.DataFrame({
        "linha_dataset": dados_experimento.index.to_numpy(),
        "experimento": experimento,
        "posicao_no_experimento": np.arange(len(dados_experimento)),
        "y": dados_experimento["y"].to_numpy(),
        "y_pred_atual": previsoes,
        "linha_origem_previsao": indices_origem,
        "preenchido_com_media": np.arange(len(dados_experimento)) < i,
        nome_feature_temporal: dislocated_ypred,
    }))
    print(f"{experimento}: {len(dados_experimento):,} linhas | média y = {media_y:.6f}")

dados_temporais_old[nome_feature_temporal] = final_dislocated_predictions
debug_temporal_old_df = pd.concat(tabelas_debug, ignore_index=True)
if dados_temporais_old.shape != (len(static_df_old), static_df_old.shape[1] + 1):
    raise ValueError("O dataset deve manter todas as linhas e ganhar somente uma coluna.")

# Salva apenas a nova feature e sua rastreabilidade, sem duplicar o CSV de 386,8 MB.
# O dataset completo acrescido da coluna 1536 fica em dados_temporais_old, na memória.
identificador_execucao = datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S_%f_UTC")
primeira_feature_dir_old = (
    EXECUCAO_ESTATICA_OLD_DIR / "primeira_feature_temporal_old" / identificador_execucao
)
primeira_feature_dir_old.mkdir(parents=True, exist_ok=False)
PRIMEIRA_FEATURE_OLD_PATH = primeira_feature_dir_old / "primeira_feature_temporal_old.csv"
debug_temporal_old_df.to_csv(PRIMEIRA_FEATURE_OLD_PATH, index=False)

# Tabela didática: a coluna 1536 da segunda linha deve repetir y_pred_atual da primeira.
display(debug_temporal_old_df[debug_temporal_old_df["experimento"] == "L1"].head(8))
print(f"Dataset estático: {static_df_old.shape}")
print(f"Dataset com uma feature temporal: {dados_temporais_old.shape}")
print(f"Nova coluna: {nome_feature_temporal} | Nenhuma linha foi removida.")
print(f"Feature e rastreabilidade salvas em: {PRIMEIRA_FEATURE_OLD_PATH}")
print("O modelo ainda NÃO foi treinado com a nova coluna; isso será a próxima etapa.")


## 12. Regressor com uma feature temporal — metodologia `_old`

A coluna `1536` foi validada e agora será incorporada à entrada do regressor. O modelo receberá as 1.536 features visuais originais mais essa primeira feature temporal, totalizando **1.537 entradas**.

A arquitetura e o treinamento permanecem iguais ao baseline: uma camada densa com 32 neurônios, uma saída para regressão, otimizador Adam, MAE, até 300 épocas, lotes de 128 e parada antecipada com paciência de 20 épocas. A divisão continua em 70% para treinamento e 30% para validação, com `random_state=42`.

No código de referência, essa etapa corresponde à iteração `i = 2`: a coluna `1536` já existe e é utilizada para treinar o novo modelo. As métricas desta etapa ainda são de validação, não de um teste independente. O modelo salvo nesta célula poderá ser usado depois para gerar a próxima coluna temporal (`1537`).

A quantidade de entradas é obtida a partir de `X_train_temporal_1_old.shape[1]`, evitando uma segunda definição manual inconsistente. O protocolo continua sendo `_old`, portanto a feature temporal mantém a associação e o vazamento deliberados da reprodução original.


In [ ]:
from pathlib import Path
from datetime import datetime, timezone
from time import perf_counter
import numpy as np
import pandas as pd
from tensorflow import keras
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_absolute_percentage_error, r2_score

# Esta célula depende da execução da célula 30.
# Ela usa as 1.536 features originais mais a coluna temporal 1536.
if "dados_temporais_old" not in globals() or "1536" not in dados_temporais_old.columns:
    raise RuntimeError(
        "Execute primeiro a célula 30 para criar dados_temporais_old e a coluna 1536."
    )

OUTPUT_DIR = Path("/content/drive/MyDrive/TCC/resultados/inceptionresnetv2")
colunas_features_temporais_old = [str(j) for j in range(1537)]
X_temporal_1_old = dados_temporais_old[colunas_features_temporais_old].astype(np.float32)
y_temporal_1_old = dados_temporais_old["y"]

if not np.isfinite(X_temporal_1_old.to_numpy()).all():
    raise ValueError("A entrada temporal contém features NaN ou infinitas.")
if not np.isfinite(y_temporal_1_old.to_numpy()).all():
    raise ValueError("O alvo y contém valores NaN ou infinitos.")

# A divisão reproduz a configuração usada no código de referência.
X_train_temporal_1_old, X_val_temporal_1_old, y_train_temporal_1_old, y_val_temporal_1_old = train_test_split(
    X_temporal_1_old,
    y_temporal_1_old,
    train_size=0.7,
    random_state=42,
)
n_features_entrada_temporal_1_old = X_train_temporal_1_old.shape[1]
print(
    f"Entrada do regressor: {n_features_entrada_temporal_1_old} features | "
    f"Treino: {X_train_temporal_1_old.shape} | "
    f"Validação: {X_val_temporal_1_old.shape}"
)

# Equivale à iteração i=2 do código original: 1536 features + uma temporal.
modelo_temporal_1_old = keras.Sequential([
    keras.layers.Input(shape=(n_features_entrada_temporal_1_old,)),
    keras.layers.Dense(32, activation="relu"),
    keras.layers.Dense(1),
])
early_stopping_temporal_1_old = keras.callbacks.EarlyStopping(
    monitor="val_loss",
    patience=20,
    verbose=0,
    mode="auto",
    baseline=None,
    restore_best_weights=True,
    start_from_epoch=0,
)
modelo_temporal_1_old.compile(
    optimizer="adam",
    loss="mean_absolute_error",
)
modelo_temporal_1_old.summary()

# A validação é usada no early stopping e nas métricas, como na versão _old.
inicio_treino_temporal_1_old = perf_counter()
historico_temporal_1_old = modelo_temporal_1_old.fit(
    X_train_temporal_1_old,
    y_train_temporal_1_old,
    epochs=300,
    batch_size=128,
    validation_data=(X_val_temporal_1_old, y_val_temporal_1_old),
    callbacks=[early_stopping_temporal_1_old],
    verbose=2,
)
tempo_treino_temporal_1_old_s = perf_counter() - inicio_treino_temporal_1_old

y_pred_temporal_1_old = modelo_temporal_1_old.predict(
    X_val_temporal_1_old,
    verbose=0,
).reshape(-1)
metricas_temporais_1_old_df = pd.DataFrame([{
    "metodologia": "old",
    "features_temporais": 1,
    "particao": "validacao",
    "n_features_entrada": n_features_entrada_temporal_1_old,
    "n_treino": len(y_train_temporal_1_old),
    "n_validacao": len(y_val_temporal_1_old),
    "MAE_mm": mean_absolute_error(y_val_temporal_1_old, y_pred_temporal_1_old),
    "MAPE_pct": 100 * mean_absolute_percentage_error(
        y_val_temporal_1_old, y_pred_temporal_1_old
    ),
    "R2": r2_score(y_val_temporal_1_old, y_pred_temporal_1_old),
    "epocas_executadas": len(historico_temporal_1_old.history["loss"]),
    "melhor_epoca": int(
        np.argmin(historico_temporal_1_old.history["val_loss"]) + 1
    ),
    "tempo_treino_s": tempo_treino_temporal_1_old_s,
}])

previsoes_temporais_1_old_df = pd.DataFrame({
    "linha_dataset": y_val_temporal_1_old.index.to_numpy(),
    "experimento": dados_temporais_old.loc[
        y_val_temporal_1_old.index, "experimento"
    ].to_numpy(),
    "y": y_val_temporal_1_old.to_numpy(),
    "y_pred": y_pred_temporal_1_old,
})
previsoes_temporais_1_old_df["erro_absoluto_mm"] = np.abs(
    previsoes_temporais_1_old_df["y"] - previsoes_temporais_1_old_df["y_pred"]
)

# Cada execução fica em uma pasta nova e pode ser comparada com o baseline estático.
if "PRIMEIRA_FEATURE_OLD_PATH" not in globals():
    raise RuntimeError("A variável PRIMEIRA_FEATURE_OLD_PATH não foi criada pela célula 30.")
identificador_execucao_temporal_1_old = datetime.now(timezone.utc).strftime(
    "%Y%m%d_%H%M%S_%f_UTC"
)
EXECUCAO_TEMPORAL_1_OLD_DIR = (
    PRIMEIRA_FEATURE_OLD_PATH.parent
    / "regressao_temporal_1_old"
    / identificador_execucao_temporal_1_old
)
EXECUCAO_TEMPORAL_1_OLD_DIR.mkdir(parents=True, exist_ok=False)
modelo_temporal_1_old.save(
    EXECUCAO_TEMPORAL_1_OLD_DIR / "modelo_temporal_1_old.keras"
)
metricas_temporais_1_old_df.to_csv(
    EXECUCAO_TEMPORAL_1_OLD_DIR / "metricas_temporal_1_old.csv",
    index=False,
)
previsoes_temporais_1_old_df.to_csv(
    EXECUCAO_TEMPORAL_1_OLD_DIR / "previsoes_validacao_temporal_1_old.csv",
    index=False,
)
historico_temporal_1_old_df = pd.DataFrame(historico_temporal_1_old.history)
historico_temporal_1_old_df.insert(
    0,
    "epoca",
    np.arange(1, len(historico_temporal_1_old_df) + 1),
)
historico_temporal_1_old_df.to_csv(
    EXECUCAO_TEMPORAL_1_OLD_DIR / "historico_temporal_1_old.csv",
    index=False,
)
pd.DataFrame({"linha_dataset": y_train_temporal_1_old.index}).to_csv(
    EXECUCAO_TEMPORAL_1_OLD_DIR / "indices_treino_temporal_1_old.csv",
    index=False,
)

display(metricas_temporais_1_old_df)
display(previsoes_temporais_1_old_df.head())
print(f"Resultados salvos em: {EXECUCAO_TEMPORAL_1_OLD_DIR}")
print(
    "Regressor temporal _old concluído. "
    "Nenhuma nova feature temporal foi gerada nesta célula."
)


## 13. Segunda feature temporal — metodologia `_old`

O regressor da etapa anterior foi treinado com 1.537 entradas. Nesta etapa, ele é usado para gerar a próxima coluna temporal, `1537`, que será acrescentada ao dataset. Como essa é a iteração `i = 2`, as duas primeiras linhas de cada experimento recebem a média de `y`; as linhas seguintes recebem as previsões deslocadas do modelo.

A célula não treina uma rede nova e não remove linhas. Ela apenas amplia `dados_temporais_old` de 1.542 para 1.543 colunas e exibe uma tabela de rastreamento para confirmar o deslocamento. Depois dessa validação, o próximo regressor terá 1.538 entradas.


In [ ]:
from datetime import datetime, timezone
import numpy as np
import pandas as pd

# Esta célula usa o modelo treinado na célula 32 para gerar a próxima coluna.
if "dados_temporais_old" not in globals() or "1536" not in dados_temporais_old.columns:
    raise RuntimeError(
        "Execute primeiro as células 30 e 32 para criar o dataset temporal e o modelo."
    )
if "modelo_temporal_1_old" not in globals():
    raise RuntimeError("O modelo temporal da célula 32 não está disponível na memória.")
if "1537" in dados_temporais_old.columns:
    raise RuntimeError("A coluna 1537 já existe; não será sobrescrita.")

# A iteração i=2 recebe as 1537 colunas anteriores e gera a coluna 1537.
i = 2
nome_feature_temporal = str(1536 + i - 1)
colunas_entrada_temporal = [str(j) for j in range(1537)]
final_dislocated_predictions_2_old = []
tabelas_debug_2_old = []
ordem_experimentos_2_old = dados_temporais_old["experimento"].drop_duplicates().tolist()

for experimento in ordem_experimentos_2_old:
    dados_experimento = dados_temporais_old[
        dados_temporais_old["experimento"] == experimento
    ].copy()
    X_experimento = dados_experimento[colunas_entrada_temporal].to_numpy(
        dtype=np.float32
    )
    if not np.isfinite(X_experimento).all():
        raise ValueError(f"Entradas inválidas no experimento {experimento}.")

    previsoes = modelo_temporal_1_old.predict(
        X_experimento,
        verbose=0,
    ).reshape(-1)
    if len(previsoes) != len(dados_experimento) or not np.isfinite(previsoes).all():
        raise ValueError(f"Previsões inválidas no experimento {experimento}.")

    # Reprodução deliberada do código original para i=2:
    # as duas primeiras linhas recebem a média de y do experimento.
    media_y = dados_experimento["y"].mean()
    dislocated_ypred = [media_y] * i
    dislocated_ypred.extend(previsoes.tolist())
    dislocated_ypred = dislocated_ypred[:len(dados_experimento)]
    final_dislocated_predictions_2_old.extend(dislocated_ypred)

    indices_origem = pd.array(
        [pd.NA] * i + dados_experimento.index.tolist(),
        dtype="Int64",
    )[:len(dados_experimento)]
    tabelas_debug_2_old.append(pd.DataFrame({
        "linha_dataset": dados_experimento.index.to_numpy(),
        "experimento": experimento,
        "posicao_no_experimento": np.arange(len(dados_experimento)),
        "y": dados_experimento["y"].to_numpy(),
        "y_pred_atual": previsoes,
        "linha_origem_previsao": indices_origem,
        "preenchido_com_media": np.arange(len(dados_experimento)) < i,
        nome_feature_temporal: dislocated_ypred,
    }))
    print(f"{experimento}: {len(dados_experimento):,} linhas | média y = {media_y:.6f}")

dados_temporais_old[nome_feature_temporal] = final_dislocated_predictions_2_old
debug_temporal_2_old_df = pd.concat(tabelas_debug_2_old, ignore_index=True)
if dados_temporais_old.shape != (len(dados_temporais_old), 1543):
    raise ValueError("O dataset deveria ter 1.543 colunas após a segunda feature temporal.")

# Salva a rastreabilidade em uma pasta própria; o dataset completo permanece na memória.
identificador_execucao_2_old = datetime.now(timezone.utc).strftime(
    "%Y%m%d_%H%M%S_%f_UTC"
)
SEGUNDA_FEATURE_OLD_DIR = (
    EXECUCAO_TEMPORAL_1_OLD_DIR
    / "segunda_feature_temporal_old"
    / identificador_execucao_2_old
)
SEGUNDA_FEATURE_OLD_DIR.mkdir(parents=True, exist_ok=False)
DEBUG_SEGUNDA_FEATURE_OLD_PATH = SEGUNDA_FEATURE_OLD_DIR / "segunda_feature_temporal_old.csv"
debug_temporal_2_old_df.to_csv(DEBUG_SEGUNDA_FEATURE_OLD_PATH, index=False)

display(debug_temporal_2_old_df[debug_temporal_2_old_df["experimento"] == "L1"].head(8))
print(f"Dataset com duas features temporais: {dados_temporais_old.shape}")
print(f"Nova coluna: {nome_feature_temporal} | Nenhuma linha foi removida.")
print(f"Rastreabilidade salva em: {DEBUG_SEGUNDA_FEATURE_OLD_PATH}")
print("A próxima etapa será treinar o regressor com 1.538 entradas.")


## 14. Regressor com duas features temporais — metodologia `_old`

Execute primeiro a etapa 13, que adiciona a segunda feature temporal (`1537`). Agora treinamos **um novo regressor** com as 1.536 features visuais mais as duas temporais (`1536` e `1537`), totalizando **1.538 entradas**. Os parâmetros de processo, o experimento e o alvo `y` não entram em `X`.

Mantemos a divisão 70%/30% com `random_state=42`, a camada densa de 32 neurônios, Adam, MAE, até 300 épocas, lotes de 128 e parada antecipada com paciência de 20 épocas. A dimensão de entrada é obtida de `X_train.shape[1]`. As métricas continuam sendo de validação, conforme a reprodução `_old`.

Esta célula **não adiciona a terceira feature** nem altera as linhas do dataset. Ela salva o regressor com duas temporais; no próximo passo usaremos suas previsões para criar a terceira coluna, deslocando-as três linhas dentro de cada experimento. Seguiremos individualmente até a quarta feature e só depois prepararemos o loop até 100.


In [ ]:
from datetime import datetime, timezone
from time import perf_counter
import numpy as np
import pandas as pd
from tensorflow import keras
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_absolute_percentage_error, r2_score

# Pré-requisitos: a etapa 13 deve ter criado e salvo a segunda feature temporal.
if "dados_temporais_old" not in globals() or "SEGUNDA_FEATURE_OLD_DIR" not in globals():
    raise RuntimeError("Execute primeiro a etapa 13: segunda feature temporal.")
if "static_df_old" not in globals() or not SEGUNDA_FEATURE_OLD_DIR.is_dir():
    raise RuntimeError("Confira o dataset estático e a pasta criada na etapa 13.")

# Só entram as features visuais e as duas temporais; y e metadados ficam fora de X.
colunas_excluidas = ["y", "current", "nozzle_speed", "mass_flow_rate", "experimento"]
colunas_entrada = dados_temporais_old.drop(columns=colunas_excluidas).columns.tolist()
if colunas_entrada != [str(j) for j in range(1538)]:
    raise ValueError("Esperadas 1.536 features visuais e somente as temporais 1536 e 1537.")
if not dados_temporais_old.index.equals(static_df_old.index):
    raise ValueError("A quantidade ou a ordem das linhas mudou desde o dataset estático.")
if not dados_temporais_old[["experimento", "y"]].equals(static_df_old[["experimento", "y"]]):
    raise ValueError("Os experimentos ou alvos foram alterados; confira as etapas anteriores.")

X = dados_temporais_old[colunas_entrada].astype(np.float32)
y = dados_temporais_old["y"]
if not np.isfinite(X.to_numpy()).all() or not np.isfinite(y.to_numpy()).all():
    raise ValueError("Há valores NaN ou infinitos nas entradas ou no alvo.")

# Mesma divisão do código de referência, sem ordenar ou remover linhas.
X_train, X_val, y_train, y_val = train_test_split(
    X, y, train_size=0.7, random_state=42
)
n_features_entrada = X_train.shape[1]
print(f"Entrada: {n_features_entrada} features (1.536 visuais + 2 temporais)")
print(f"Treino: {X_train.shape} | Validação: {X_val.shape}")

# Novo modelo: corresponde ao treinamento da iteração i=3 do código original.
modelo_temporal_2_old = keras.Sequential([
    keras.layers.Input(shape=(n_features_entrada,)),
    keras.layers.Dense(32, activation="relu"),
    keras.layers.Dense(1),
])
early_stopping = keras.callbacks.EarlyStopping(
    monitor="val_loss", patience=20, verbose=0, mode="auto",
    baseline=None, restore_best_weights=True, start_from_epoch=0,
)
modelo_temporal_2_old.compile(optimizer="adam", loss="mean_absolute_error")
modelo_temporal_2_old.summary()

inicio_treino = perf_counter()
historico_temporal_2_old = modelo_temporal_2_old.fit(
    X_train, y_train,
    epochs=300, batch_size=128,
    validation_data=(X_val, y_val),
    callbacks=[early_stopping], verbose=2,
)
tempo_treino = perf_counter() - inicio_treino

# A validação também orienta o early stopping: não é um teste independente.
y_pred = modelo_temporal_2_old.predict(X_val, verbose=0).reshape(-1)
metricas_temporais_2_old_df = pd.DataFrame([{
    "metodologia": "old",
    "features_temporais": 2,
    "particao": "validacao",
    "n_features_entrada": n_features_entrada,
    "n_treino": len(y_train),
    "n_validacao": len(y_val),
    "MAE_mm": mean_absolute_error(y_val, y_pred),
    "MAPE_pct": 100 * mean_absolute_percentage_error(y_val, y_pred),
    "R2": r2_score(y_val, y_pred),
    "epocas_executadas": len(historico_temporal_2_old.history["loss"]),
    "melhor_epoca": int(np.argmin(historico_temporal_2_old.history["val_loss"]) + 1),
    "tempo_treino_s": tempo_treino,
}])
previsoes_validacao = pd.DataFrame({
    "linha_dataset": y_val.index.to_numpy(),
    "experimento": dados_temporais_old.loc[y_val.index, "experimento"].to_numpy(),
    "y": y_val.to_numpy(),
    "y_pred": y_pred,
    "erro_absoluto_mm": np.abs(y_val.to_numpy() - y_pred),
})

# Uma pasta nova por execução preserva os modelos e resultados anteriores.
identificador_execucao = datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S_%f_UTC")
EXECUCAO_TEMPORAL_2_OLD_DIR = (
    SEGUNDA_FEATURE_OLD_DIR / "regressao_temporal_2_old" / identificador_execucao
)
EXECUCAO_TEMPORAL_2_OLD_DIR.mkdir(parents=True, exist_ok=False)
modelo_temporal_2_old.save(EXECUCAO_TEMPORAL_2_OLD_DIR / "modelo_temporal_2_old.keras")
metricas_temporais_2_old_df.to_csv(
    EXECUCAO_TEMPORAL_2_OLD_DIR / "metricas_temporal_2_old.csv", index=False
)
previsoes_validacao.to_csv(
    EXECUCAO_TEMPORAL_2_OLD_DIR / "previsoes_validacao_temporal_2_old.csv", index=False
)
pd.DataFrame(historico_temporal_2_old.history).rename_axis("epoca_base_zero").to_csv(
    EXECUCAO_TEMPORAL_2_OLD_DIR / "historico_temporal_2_old.csv"
)
pd.DataFrame({"linha_dataset": y_train.index}).to_csv(
    EXECUCAO_TEMPORAL_2_OLD_DIR / "indices_treino_temporal_2_old.csv", index=False
)

# Visualiza as duas colunas existentes; nenhuma terceira feature é criada aqui.
display(dados_temporais_old.loc[
    dados_temporais_old["experimento"] == "L1", ["experimento", "y", "1536", "1537"]
].head(8))
display(metricas_temporais_2_old_df)
display(previsoes_validacao.head())
print(f"Dataset mantido com duas features temporais: {dados_temporais_old.shape}")
print(f"Resultados salvos em: {EXECUCAO_TEMPORAL_2_OLD_DIR}")
print("Próximo passo: usar este modelo para gerar a terceira feature temporal (1538).")


## 15. Terceira feature temporal — metodologia `_old`

Usamos o `modelo_temporal_2_old`, treinado na etapa 14 com **1.536 features visuais + duas temporais**, para gerar novas previsões para cada experimento. Não treinamos outra rede nesta etapa.

A nova coluna será **`1538`**. Como agora `i = 3`, preenchemos as três primeiras linhas de cada experimento com a média de seu `y` e deslocamos as previsões três posições. Assim, a linha de posição 3 recebe a previsão da posição 0; a posição 4 recebe a previsão da posição 1, e assim por diante. São previsões do modelo recém-treinado, não apenas outra cópia deslocada das previsões do modelo estático.

Preservamos a ordem, todas as linhas e as duas features temporais anteriores. O dataset passa de **(26634, 1543)** para **(26634, 1544)**: 1.536 features visuais, três temporais e cinco colunas de alvo/metadados. A tabela mostra as três colunas temporais lado a lado e a origem dos valores da nova coluna.

Como estamos reproduzindo o método `_old`, a média ainda usa todo o alvo do experimento, e deslocar linhas não garante deslocar frames cronológicos. As correções ficam para o template `_new`. Depois da conferência, o próximo regressor terá **1.539 entradas**.


In [ ]:
from datetime import datetime, timezone
import numpy as np
import pandas as pd

# Usa somente o regressor já treinado na etapa 14; não há model.fit nesta célula.
if "dados_temporais_old" not in globals() or "modelo_temporal_2_old" not in globals():
    raise RuntimeError("Execute primeiro as etapas 13 e 14.")
if "EXECUCAO_TEMPORAL_2_OLD_DIR" not in globals():
    raise RuntimeError("A pasta de resultados do regressor com duas temporais não foi definida.")
if not (EXECUCAO_TEMPORAL_2_OLD_DIR / "modelo_temporal_2_old.keras").is_file():
    raise RuntimeError("Confira se a etapa 14 terminou e salvou o modelo.")

# Terceira iteração: a entrada tem duas temporais e será criada apenas a terceira.
i = 3
nome_feature_temporal = str(1536 + i - 1)  # 1538
if nome_feature_temporal in dados_temporais_old.columns:
    raise RuntimeError("A terceira feature (1538) já existe; não será sobrescrita.")
colunas_excluidas = ["y", "current", "nozzle_speed", "mass_flow_rate", "experimento"]
colunas_entrada = dados_temporais_old.drop(columns=colunas_excluidas).columns.tolist()
if colunas_entrada != [str(j) for j in range(1538)]:
    raise ValueError("O dataset deve conter as features 0 a 1537, nessa ordem.")
if modelo_temporal_2_old.input_shape[-1] != len(colunas_entrada):
    raise ValueError("O modelo não corresponde às 1.538 entradas esperadas.")
if dados_temporais_old.empty or not dados_temporais_old.index.is_unique:
    raise ValueError("O dataset está vazio ou possui índices duplicados.")
if dados_temporais_old["experimento"].isna().any():
    raise ValueError("Há linhas sem identificação do experimento.")

# A concatenação pressupõe blocos contíguos, como nas etapas anteriores.
# Apenas verifica a ordem: não ordena os frames nem modifica a metodologia _old.
ordem_experimentos = dados_temporais_old["experimento"].drop_duplicates().tolist()
ordem_blocos = dados_temporais_old.loc[
    dados_temporais_old["experimento"].ne(dados_temporais_old["experimento"].shift()),
    "experimento",
].tolist()
if ordem_blocos != ordem_experimentos:
    raise ValueError("Os experimentos precisam estar em blocos contíguos.")
n_linhas_antes, n_colunas_antes = dados_temporais_old.shape
final_dislocated_predictions = []
tabelas_debug = []

for experimento in ordem_experimentos:
    dados_experimento = dados_temporais_old[
        dados_temporais_old["experimento"] == experimento
    ].copy()
    X_experimento = dados_experimento[colunas_entrada].to_numpy(dtype=np.float32)
    if not np.isfinite(X_experimento).all() or not np.isfinite(dados_experimento["y"]).all():
        raise ValueError(f"Entradas ou alvos inválidos em {experimento}.")

    # As previsões agora vêm do modelo treinado COM DUAS features temporais.
    previsoes = modelo_temporal_2_old.predict(X_experimento, verbose=0).reshape(-1)
    if len(previsoes) != len(dados_experimento) or not np.isfinite(previsoes).all():
        raise ValueError(f"Previsões inválidas em {experimento}.")

    # Reprodução _old: três médias no início e previsões deslocadas três linhas.
    # A média usa todo o y do experimento, inclusive a parte de validação.
    media_y = dados_experimento["y"].mean()
    dislocated_ypred = [media_y] * i
    dislocated_ypred.extend(previsoes.tolist())
    dislocated_ypred = dislocated_ypred[:len(dados_experimento)]
    final_dislocated_predictions.extend(dislocated_ypred)

    indices_origem = pd.array(
        [pd.NA] * i + dados_experimento.index.tolist(), dtype="Int64"
    )[:len(dados_experimento)]
    tabelas_debug.append(pd.DataFrame({
        "linha_dataset": dados_experimento.index.to_numpy(),
        "experimento": experimento,
        "posicao_no_experimento": np.arange(len(dados_experimento)),
        "y": dados_experimento["y"].to_numpy(),
        "1536": dados_experimento["1536"].to_numpy(),
        "1537": dados_experimento["1537"].to_numpy(),
        "y_pred_atual": previsoes,
        "linha_origem_previsao": indices_origem,
        "preenchido_com_media": np.arange(len(dados_experimento)) < i,
        nome_feature_temporal: dislocated_ypred,
    }))
    print(f"{experimento}: {len(dados_experimento):,} linhas | média y = {media_y:.6f}")

debug_temporal_3_old_df = pd.concat(tabelas_debug, ignore_index=True)
if len(final_dislocated_predictions) != n_linhas_antes:
    raise ValueError("A nova coluna não tem a mesma quantidade de linhas do dataset.")
if not np.array_equal(debug_temporal_3_old_df["linha_dataset"].to_numpy(), dados_temporais_old.index.to_numpy()):
    raise ValueError("A ordem da tabela de rastreamento difere da ordem do dataset.")

# Salva primeiro a tabela compacta; não duplica as 1.536 features visuais em outro CSV.
identificador_execucao = datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S_%f_UTC")
TERCEIRA_FEATURE_OLD_DIR = (
    EXECUCAO_TEMPORAL_2_OLD_DIR / "terceira_feature_temporal_old" / identificador_execucao
)
TERCEIRA_FEATURE_OLD_DIR.mkdir(parents=True, exist_ok=False)
DEBUG_TERCEIRA_FEATURE_OLD_PATH = TERCEIRA_FEATURE_OLD_DIR / "terceira_feature_temporal_old.csv"
debug_temporal_3_old_df.to_csv(DEBUG_TERCEIRA_FEATURE_OLD_PATH, index=False)

# Acrescenta somente a coluna 1538; todas as outras colunas e linhas permanecem.
dados_temporais_old[nome_feature_temporal] = final_dislocated_predictions
if dados_temporais_old.shape != (n_linhas_antes, n_colunas_antes + 1):
    raise ValueError("O dataset deveria ganhar somente uma coluna, sem remover linhas.")

display(debug_temporal_3_old_df.loc[
    debug_temporal_3_old_df["experimento"] == "L1"
].head(8))
print(f"Dataset antes: {(n_linhas_antes, n_colunas_antes)}")
print(f"Dataset com três features temporais: {dados_temporais_old.shape}")
print(f"Nova coluna: {nome_feature_temporal} | Nenhuma linha foi removida.")
print(f"Rastreabilidade salva em: {DEBUG_TERCEIRA_FEATURE_OLD_PATH}")
print("Na posição 3, a coluna 1538 deve repetir y_pred_atual da posição 0 do mesmo experimento.")
print("O regressor com TRÊS temporais (1.539 entradas) ainda não foi treinado; será o próximo passo.")


## 16. Regressor com três features temporais — metodologia `_old`

A etapa 15 adicionou a coluna `1538`. Agora treinamos **um novo regressor** usando as 1.536 features visuais e as três temporais (`1536`, `1537` e `1538`): são **1.539 entradas**. O alvo `y`, os parâmetros de processo e a identificação do experimento continuam fora de `X`.

Mantemos a mesma divisão 70%/30% (`random_state=42`) e o mesmo treinamento: camada densa com 32 neurônios e ReLU, uma saída de regressão, Adam, MAE, até 300 épocas, lotes de 128 e parada antecipada com paciência de 20 épocas. A dimensão de entrada é obtida de `X_train.shape[1]`. No laço original, esse treinamento corresponde ao início da iteração `i = 4`.

A célula apresenta as métricas de **validação**, salva o modelo e seus resultados em uma nova pasta e **não altera o dataset nem cria a quarta feature**. Usaremos as previsões desse modelo para gerar a coluna `1539` no próximo passo. A metodologia `_old`, incluindo suas limitações, permanece preservada.


In [ ]:
from datetime import datetime, timezone
from time import perf_counter
import numpy as np
import pandas as pd
from tensorflow import keras
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_absolute_percentage_error, r2_score

# Pré-requisitos: a etapa 15 deve ter criado e salvo a terceira feature temporal.
if "dados_temporais_old" not in globals() or "TERCEIRA_FEATURE_OLD_DIR" not in globals():
    raise RuntimeError("Execute primeiro a etapa 15: terceira feature temporal.")
if "static_df_old" not in globals() or not TERCEIRA_FEATURE_OLD_DIR.is_dir():
    raise RuntimeError("Confira o dataset estático e a pasta criada na etapa 15.")

# Só entram as features visuais e as três temporais; y e metadados ficam fora de X.
colunas_excluidas = ["y", "current", "nozzle_speed", "mass_flow_rate", "experimento"]
colunas_entrada = dados_temporais_old.drop(columns=colunas_excluidas).columns.tolist()
if colunas_entrada != [str(j) for j in range(1539)]:
    raise ValueError("Esperadas 1.536 features visuais e somente as temporais 1536, 1537 e 1538.")
if not dados_temporais_old.index.equals(static_df_old.index):
    raise ValueError("A quantidade ou a ordem das linhas mudou desde o dataset estático.")
if not dados_temporais_old[["experimento", "y"]].equals(static_df_old[["experimento", "y"]]):
    raise ValueError("Os experimentos ou alvos foram alterados; confira as etapas anteriores.")

X = dados_temporais_old[colunas_entrada].astype(np.float32)
y = dados_temporais_old["y"]
if not np.isfinite(X.to_numpy()).all() or not np.isfinite(y.to_numpy()).all():
    raise ValueError("Há valores NaN ou infinitos nas entradas ou no alvo.")

# Mesma divisão do código de referência, sem ordenar ou remover linhas.
X_train, X_val, y_train, y_val = train_test_split(
    X, y, train_size=0.7, random_state=42
)
n_features_entrada = X_train.shape[1]
print(f"Entrada: {n_features_entrada} features (1.536 visuais + 3 temporais)")
print(f"Treino: {X_train.shape} | Validação: {X_val.shape}")

# Novo modelo: corresponde ao treinamento da iteração i=4 do código original.
modelo_temporal_3_old = keras.Sequential([
    keras.layers.Input(shape=(n_features_entrada,)),
    keras.layers.Dense(32, activation="relu"),
    keras.layers.Dense(1),
])
early_stopping = keras.callbacks.EarlyStopping(
    monitor="val_loss", patience=20, verbose=0, mode="auto",
    baseline=None, restore_best_weights=True, start_from_epoch=0,
)
modelo_temporal_3_old.compile(optimizer="adam", loss="mean_absolute_error")
modelo_temporal_3_old.summary()

inicio_treino = perf_counter()
historico_temporal_3_old = modelo_temporal_3_old.fit(
    X_train, y_train,
    epochs=300, batch_size=128,
    validation_data=(X_val, y_val),
    callbacks=[early_stopping], verbose=2,
)
tempo_treino = perf_counter() - inicio_treino

# A validação também orienta o early stopping: não é um teste independente.
y_pred = modelo_temporal_3_old.predict(X_val, verbose=0).reshape(-1)
metricas_temporais_3_old_df = pd.DataFrame([{
    "metodologia": "old",
    "features_temporais": 3,
    "particao": "validacao",
    "n_features_entrada": n_features_entrada,
    "n_treino": len(y_train),
    "n_validacao": len(y_val),
    "MAE_mm": mean_absolute_error(y_val, y_pred),
    "MAPE_pct": 100 * mean_absolute_percentage_error(y_val, y_pred),
    "R2": r2_score(y_val, y_pred),
    "epocas_executadas": len(historico_temporal_3_old.history["loss"]),
    "melhor_epoca": int(np.argmin(historico_temporal_3_old.history["val_loss"]) + 1),
    "tempo_treino_s": tempo_treino,
}])
previsoes_validacao = pd.DataFrame({
    "linha_dataset": y_val.index.to_numpy(),
    "experimento": dados_temporais_old.loc[y_val.index, "experimento"].to_numpy(),
    "y": y_val.to_numpy(),
    "y_pred": y_pred,
    "erro_absoluto_mm": np.abs(y_val.to_numpy() - y_pred),
})

# Uma pasta nova por execução preserva os modelos e resultados anteriores.
identificador_execucao = datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S_%f_UTC")
EXECUCAO_TEMPORAL_3_OLD_DIR = (
    TERCEIRA_FEATURE_OLD_DIR / "regressao_temporal_3_old" / identificador_execucao
)
EXECUCAO_TEMPORAL_3_OLD_DIR.mkdir(parents=True, exist_ok=False)
modelo_temporal_3_old.save(EXECUCAO_TEMPORAL_3_OLD_DIR / "modelo_temporal_3_old.keras")
metricas_temporais_3_old_df.to_csv(
    EXECUCAO_TEMPORAL_3_OLD_DIR / "metricas_temporal_3_old.csv", index=False
)
previsoes_validacao.to_csv(
    EXECUCAO_TEMPORAL_3_OLD_DIR / "previsoes_validacao_temporal_3_old.csv", index=False
)
pd.DataFrame(historico_temporal_3_old.history).rename_axis("epoca_base_zero").to_csv(
    EXECUCAO_TEMPORAL_3_OLD_DIR / "historico_temporal_3_old.csv"
)
pd.DataFrame({"linha_dataset": y_train.index}).to_csv(
    EXECUCAO_TEMPORAL_3_OLD_DIR / "indices_treino_temporal_3_old.csv", index=False
)

# Visualiza as três colunas existentes; nenhuma quarta feature é criada aqui.
display(dados_temporais_old.loc[
    dados_temporais_old["experimento"] == "L1", ["experimento", "y", "1536", "1537", "1538"]
].head(8))
display(metricas_temporais_3_old_df)
display(previsoes_validacao.head())
print(f"Dataset mantido com três features temporais: {dados_temporais_old.shape}")
print(f"Resultados salvos em: {EXECUCAO_TEMPORAL_3_OLD_DIR}")
print("Próximo passo: usar este modelo para gerar a quarta feature temporal (1539).")


## 17. Laço temporal até 100 — metodologia _old

Esta versão do laço começa diretamente no dataset estático: as colunas 0 a 1535 são as features visuais e ainda não existe nenhuma feature temporal. A primeira rodada usa o modelo estático para gerar a coluna 1536. Em seguida, o laço treina um modelo com uma temporal, gera 1537, e repete o processo até 1635.

A sequência é:

1. Carregar o dataset estático e o modelo estático.
2. Gerar a próxima feature temporal usando o modelo atual.
3. Acrescentar essa coluna ao dataset.
4. Treinar o modelo que será usado na próxima rodada.
5. Salvar a feature, o modelo, as métricas e o checkpoint.

O dataset conserva as 26.634 linhas. Em cada iteração, as i primeiras posições de cada experimento recebem a média de y, e as previsões são deslocadas i posições, preservando a regra _old. Ao final haverá 100 features temporais (1536 a 1635) e 1.641 colunas no total.

Se o Colab desconectar, copie o caminho impresso para PASTA_RETOMADA e execute novamente. O código reconstruirá o dataset estático e carregará as features já concluídas, sem exigir que dados_temporais_old exista na memória. A 100ª feature é gerada pelo modelo treinado com 99 temporais; não é treinado um modelo adicional com 100 temporais.

In [ ]:
from pathlib import Path
from datetime import datetime, timezone
from time import perf_counter
import json
import numpy as np
import pandas as pd
from tensorflow import keras
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_absolute_percentage_error, r2_score

TOTAL_TEMPORAIS = 100
# None inicia diretamente no dataset estático.
# Para retomar, cole aqui o caminho impresso em EXECUCAO_LOOP_OLD_DIR.
PASTA_RETOMADA = None

def salvar_json_atomico(conteudo, caminho):
    temporario = caminho.with_suffix(".tmp")
    temporario.write_text(json.dumps(conteudo, indent=2), encoding="utf-8")
    temporario.replace(caminho)

if "OUTPUT_DIR" not in globals():
    OUTPUT_DIR = Path("/content/drive/MyDrive/TCC/resultados/inceptionresnetv2")
else:
    OUTPUT_DIR = Path(OUTPUT_DIR)
if "STATIC_DATASET_OLD_PATH" not in globals():
    STATIC_DATASET_OLD_PATH = OUTPUT_DIR / "static_inceptionresnetv2_merged_df_old.csv"
else:
    STATIC_DATASET_OLD_PATH = Path(STATIC_DATASET_OLD_PATH)
if "MODELO_ESTATICO_OLD_PATH" not in globals():
    candidatos = sorted(OUTPUT_DIR.glob("regressao_estatica_old/**/modelo_estatico_old.keras"))
    if not candidatos:
        raise FileNotFoundError("Não encontrei modelo_estatico_old.keras.")
    MODELO_ESTATICO_OLD_PATH = candidatos[-1]
else:
    MODELO_ESTATICO_OLD_PATH = Path(MODELO_ESTATICO_OLD_PATH)

COLUNAS_METADADOS = ["y", "current", "nozzle_speed", "mass_flow_rate", "experimento"]
COLUNAS_VISUAIS = [str(j) for j in range(1536)]

def carregar_estatico(caminho):
    df = pd.read_csv(Path(caminho))
    if df.drop(columns=COLUNAS_METADADOS).columns.tolist() != COLUNAS_VISUAIS:
        raise ValueError("O dataset de entrada deve ser estático: features 0 a 1535.")
    if df.empty or df.index.has_duplicates:
        raise ValueError("Dataset vazio ou com índices duplicados.")
    if df["experimento"].isna().any() or not np.isfinite(df["y"]).all():
        raise ValueError("Há experimento ausente ou y inválido.")
    ordem = df["experimento"].drop_duplicates().tolist()
    blocos = df.loc[df["experimento"].ne(df["experimento"].shift()), "experimento"].tolist()
    if ordem != blocos:
        raise ValueError("Os experimentos precisam estar em blocos contíguos.")
    return df

if PASTA_RETOMADA is None:
    static_df_loop_old = carregar_estatico(STATIC_DATASET_OLD_PATH)
    if not MODELO_ESTATICO_OLD_PATH.is_file():
        raise FileNotFoundError(f"Modelo estático não encontrado: {MODELO_ESTATICO_OLD_PATH}")
    identificador = datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S_%f_UTC")
    EXECUCAO_LOOP_OLD_DIR = OUTPUT_DIR / "loop_desde_estatico_old" / identificador
    EXECUCAO_LOOP_OLD_DIR.mkdir(parents=True, exist_ok=False)
    configuracao = {
        "origem": "dataset_estatico",
        "dataset_estatico": str(STATIC_DATASET_OLD_PATH),
        "modelo_estatico": str(MODELO_ESTATICO_OLD_PATH),
        "n_features_visuais": 1536,
        "total_temporais": TOTAL_TEMPORAIS,
        "metodologia": "old",
    }
    np.savez_compressed(
        EXECUCAO_LOOP_OLD_DIR / "base_estatica.npz",
        linha_dataset=static_df_loop_old.index.to_numpy(),
        y=static_df_loop_old["y"].to_numpy(),
        experimento=static_df_loop_old["experimento"].to_numpy(dtype=str),
    )
    salvar_json_atomico(configuracao, EXECUCAO_LOOP_OLD_DIR / "configuracao.json")
    salvar_json_atomico({"ultima_feature_gerada": 0}, EXECUCAO_LOOP_OLD_DIR / "progresso.json")
else:
    EXECUCAO_LOOP_OLD_DIR = Path(PASTA_RETOMADA)
    configuracao = json.loads(
        (EXECUCAO_LOOP_OLD_DIR / "configuracao.json").read_text(encoding="utf-8")
    )
    if configuracao.get("origem") != "dataset_estatico":
        raise ValueError("A pasta de retomada não veio do dataset estático.")
    if configuracao["total_temporais"] != TOTAL_TEMPORAIS:
        raise ValueError("O checkpoint possui outro total de temporais.")

print(f"Pasta desta execução: {EXECUCAO_LOOP_OLD_DIR}", flush=True)
progresso_path = EXECUCAO_LOOP_OLD_DIR / "progresso.json"
ultima_feature = json.loads(progresso_path.read_text(encoding="utf-8"))["ultima_feature_gerada"]
if not 0 <= ultima_feature <= TOTAL_TEMPORAIS:
    raise ValueError("Checkpoint inválido.")

static_df_loop_old = carregar_estatico(configuracao["dataset_estatico"])
with np.load(EXECUCAO_LOOP_OLD_DIR / "base_estatica.npz", allow_pickle=False) as base:
    alinhado = (
        len(static_df_loop_old) == len(base["y"])
        and np.array_equal(static_df_loop_old.index.to_numpy(), base["linha_dataset"])
        and np.allclose(static_df_loop_old["y"], base["y"], rtol=0, atol=1e-6)
        and np.array_equal(
            static_df_loop_old["experimento"].to_numpy(dtype=str), base["experimento"]
        )
    )
if not alinhado:
    raise ValueError("O dataset estático não corresponde ao checkpoint.")

dados_loop_old = static_df_loop_old.copy()
for numero in range(1, ultima_feature + 1):
    caminho = EXECUCAO_LOOP_OLD_DIR / f"feature_{numero:03d}.npy"
    feature = np.load(caminho, allow_pickle=False)
    if feature.shape != (len(dados_loop_old),) or not np.isfinite(feature).all():
        raise ValueError(f"Feature salva inválida: {numero}.")
    dados_loop_old[str(1536 + numero - 1)] = feature

ordem_experimentos = dados_loop_old["experimento"].drop_duplicates().tolist()
n_linhas = len(dados_loop_old)
y_loop = dados_loop_old["y"]
metricas_acumuladas = []
for numero in range(0, ultima_feature):
    caminho = EXECUCAO_LOOP_OLD_DIR / f"metricas_com_{numero:03d}_temporais.csv"
    if not caminho.is_file():
        raise FileNotFoundError(f"Métrica não encontrada: {caminho}")
    metricas_acumuladas.append(pd.read_csv(caminho))

for i in range(ultima_feature + 1, TOTAL_TEMPORAIS + 1):
    n_temporais_entrada = i - 1
    colunas_entrada = [str(j) for j in range(1536 + n_temporais_entrada)]
    X_loop = dados_loop_old[colunas_entrada].astype(np.float32)
    caminho_modelo = (
        Path(configuracao["modelo_estatico"])
        if i == 1
        else EXECUCAO_LOOP_OLD_DIR / f"modelo_com_{n_temporais_entrada:03d}_temporais.keras"
    )
    if not caminho_modelo.is_file():
        raise FileNotFoundError(f"Modelo necessário não encontrado: {caminho_modelo}")
    modelo_atual = keras.models.load_model(caminho_modelo, compile=False)
    if modelo_atual.input_shape[-1] != X_loop.shape[1]:
        raise ValueError("Dimensão do modelo diferente da entrada atual.")

    caminho_metricas_atual = (
        EXECUCAO_LOOP_OLD_DIR / f"metricas_com_{n_temporais_entrada:03d}_temporais.csv"
    )
    if caminho_metricas_atual.is_file():
        metricas_atual = pd.read_csv(caminho_metricas_atual)
    else:
        X_train_atual, X_val_atual, y_train_atual, y_val_atual = train_test_split(
            X_loop, y_loop, train_size=0.7, random_state=42
        )
        pred_atual = modelo_atual.predict(X_val_atual, verbose=0).reshape(-1)
        metricas_atual = pd.DataFrame([{
            "metodologia": "old", "features_temporais": n_temporais_entrada,
            "particao": "validacao", "n_features_entrada": X_loop.shape[1],
            "n_treino": len(y_train_atual), "n_validacao": len(y_val_atual),
            "MAE_mm": mean_absolute_error(y_val_atual, pred_atual),
            "MAPE_pct": 100 * mean_absolute_percentage_error(y_val_atual, pred_atual),
            "R2": r2_score(y_val_atual, pred_atual),
            "epocas_executadas": np.nan, "melhor_epoca": np.nan, "tempo_treino_s": np.nan,
        }])
        metricas_atual.to_csv(caminho_metricas_atual, index=False)
    metricas_acumuladas.append(metricas_atual)

    print(
        f"Gerando temporal {i}/{TOTAL_TEMPORAIS} | modelo com {n_temporais_entrada} temporais",
        flush=True,
    )
    final_dislocated_predictions = []
    for experimento in ordem_experimentos:
        dados_experimento = dados_loop_old[
            dados_loop_old["experimento"] == experimento
        ]
        previsoes = modelo_atual.predict(
            X_loop.loc[dados_experimento.index], verbose=0
        ).reshape(-1)
        media_y = dados_experimento["y"].mean()
        deslocadas = ([media_y] * i + previsoes.tolist())[:len(dados_experimento)]
        final_dislocated_predictions.extend(deslocadas)

    nova_feature = np.asarray(final_dislocated_predictions, dtype=np.float64)
    if nova_feature.shape != (n_linhas,) or not np.isfinite(nova_feature).all():
        raise ValueError("A nova feature tem tamanho ou valores inválidos.")
    np.save(EXECUCAO_LOOP_OLD_DIR / f"feature_{i:03d}.npy", nova_feature)
    dados_loop_old[str(1536 + i - 1)] = nova_feature

    if i < TOTAL_TEMPORAIS:
        # Treina o modelo que será usado para gerar a próxima feature.
        X_proximo = dados_loop_old[
            [str(j) for j in range(1536 + i)]
        ].astype(np.float32)
        X_train, X_val, y_train, y_val = train_test_split(
            X_proximo, y_loop, train_size=0.7, random_state=42
        )
        keras.backend.clear_session()
        modelo_proximo = keras.Sequential([
            keras.layers.Input(shape=(X_train.shape[1],)),
            keras.layers.Dense(32, activation="relu"),
            keras.layers.Dense(1),
        ])
        modelo_proximo.compile(optimizer="adam", loss="mean_absolute_error")
        early_stopping = keras.callbacks.EarlyStopping(
            monitor="val_loss", patience=20, restore_best_weights=True
        )
        inicio = perf_counter()
        historico = modelo_proximo.fit(
            X_train, y_train, epochs=300, batch_size=128,
            validation_data=(X_val, y_val), callbacks=[early_stopping], verbose=0
        )
        tempo_treino = perf_counter() - inicio
        pred = modelo_proximo.predict(X_val, verbose=0).reshape(-1)
        metricas_proximo = pd.DataFrame([{
            "metodologia": "old", "features_temporais": i,
            "particao": "validacao", "n_features_entrada": X_train.shape[1],
            "n_treino": len(y_train), "n_validacao": len(y_val),
            "MAE_mm": mean_absolute_error(y_val, pred),
            "MAPE_pct": 100 * mean_absolute_percentage_error(y_val, pred),
            "R2": r2_score(y_val, pred),
            "epocas_executadas": len(historico.history["loss"]),
            "melhor_epoca": int(np.argmin(historico.history["val_loss"]) + 1),
            "tempo_treino_s": tempo_treino,
        }])
        sufixo = f"com_{i:03d}_temporais"
        modelo_proximo.save(EXECUCAO_LOOP_OLD_DIR / f"modelo_{sufixo}.keras")
        metricas_proximo.to_csv(
            EXECUCAO_LOOP_OLD_DIR / f"metricas_{sufixo}.csv", index=False
        )
        pd.DataFrame(historico.history).to_csv(
            EXECUCAO_LOOP_OLD_DIR / f"historico_{sufixo}.csv", index=False
        )
        metricas_acumuladas.append(metricas_proximo)
        del X_proximo, X_train, X_val, modelo_proximo

    metricas_loop_old_df = pd.concat(metricas_acumuladas, ignore_index=True)
    metricas_loop_old_df.to_csv(
        EXECUCAO_LOOP_OLD_DIR / "metricas_loop_old.csv", index=False
    )
    salvar_json_atomico({"ultima_feature_gerada": i}, progresso_path)
    print(
        f"Concluída {i}/{TOTAL_TEMPORAIS} | "
        f"MAE do modelo atual: {metricas_atual.iloc[0]['MAE_mm']:.6f} mm | "
        f"dataset: {dados_loop_old.shape}", flush=True
    )
    del X_loop, modelo_atual

metricas_loop_old_df = pd.concat(metricas_acumuladas, ignore_index=True)
metricas_loop_old_df.to_csv(
    EXECUCAO_LOOP_OLD_DIR / "metricas_loop_old.csv", index=False
)
if dados_loop_old.shape != (n_linhas, 1536 + TOTAL_TEMPORAIS + 5):
    raise ValueError("O dataset final não tem o formato esperado.")
display(metricas_loop_old_df.tail(10))
display(dados_loop_old.loc[
    dados_loop_old["experimento"] == "L1",
    ["experimento", "y", "1536", "1537", "1538", "1539", "1635"],
].head(8))
print(f"Finalizado: {dados_loop_old.shape} | {TOTAL_TEMPORAIS} features temporais.")
print(f"Resultados e checkpoint: {EXECUCAO_LOOP_OLD_DIR}")
print("A 100ª feature foi gerada pelo modelo treinado com 99 temporais.")
print("Não foi treinado um modelo adicional com todas as 100 temporais.")


In [ ]:
# Validação final do dataset com 100 features temporais
if "dados_loop_old" not in globals():
    raise RuntimeError("Execute primeiro a célula do laço temporal até 100.")

features_estaticas = [str(i) for i in range(1536)]
features_temporais = [str(1536 + i) for i in range(100)]
colunas_metadados_validacao = [
    "y",
    "current",
    "nozzle_speed",
    "mass_flow_rate",
    "experimento",
]

features_esperadas = features_estaticas + features_temporais
features_presentes = dados_loop_old.drop(
    columns=colunas_metadados_validacao
).columns.tolist()

print("Shape final:", dados_loop_old.shape)
print("Features esperadas:", len(features_esperadas))
print("Features presentes:", len(features_presentes))

if dados_loop_old.shape[0] != 26634:
    raise ValueError(
        f"Número inesperado de linhas: {dados_loop_old.shape[0]}"
    )

if features_presentes != features_esperadas:
    faltantes = [c for c in features_esperadas if c not in features_presentes]
    extras = [c for c in features_presentes if c not in features_esperadas]
    raise ValueError(
        f"Features incompatíveis. Faltantes: {faltantes}; extras: {extras}"
    )

if not dados_loop_old[colunas_metadados_validacao].columns.equals(
    pd.Index(colunas_metadados_validacao)
):
    raise ValueError("As colunas de metadados esperadas não foram encontradas.")

matriz_features = dados_loop_old[features_esperadas].to_numpy(dtype=np.float32)
if not np.isfinite(matriz_features).all():
    raise ValueError("Há valores NaN ou infinitos nas features.")

if not np.isfinite(dados_loop_old["y"].to_numpy()).all():
    raise ValueError("Há valores NaN ou infinitos no alvo y.")

print("Colunas faltantes: []")
print("Valores ausentes ou infinitos nas features: 0")
print("Validação concluída com sucesso.")

display(
    dados_loop_old[
        ["experimento", "y", "1536", "1537", "1635"]
    ].head(8)
)


In [ ]:
# Regressor final com as 100 features temporais — metodologia _old
from pathlib import Path
from datetime import datetime, timezone
from time import perf_counter
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow import keras
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    mean_absolute_error,
    mean_absolute_percentage_error,
    r2_score,
)

# Reutiliza o dataset em memória quando o runtime ainda está conectado.
# Caso contrário, reconstrói o dataset a partir do checkpoint salvo no Drive.
if "OUTPUT_DIR" not in globals():
    OUTPUT_DIR = Path(
        "/content/drive/MyDrive/TCC/resultados/inceptionresnetv2"
    )
else:
    OUTPUT_DIR = Path(OUTPUT_DIR)

if "dados_loop_old" in globals():
    dados_regressao_final_old = dados_loop_old.copy()
    if "EXECUCAO_LOOP_OLD_DIR" not in globals():
        candidatos_loop = sorted(
            OUTPUT_DIR.glob("loop_desde_estatico_old/*")
        )
        if not candidatos_loop:
            raise FileNotFoundError(
                "Nenhum checkpoint do laço temporal foi encontrado."
            )
        EXECUCAO_LOOP_OLD_DIR = candidatos_loop[-1]
else:
    candidatos_loop = sorted(
        OUTPUT_DIR.glob("loop_desde_estatico_old/*")
    )
    if not candidatos_loop:
        raise FileNotFoundError(
            "Não encontrei o checkpoint do laço temporal."
        )

    EXECUCAO_LOOP_OLD_DIR = candidatos_loop[-1]
    caminho_estatico = (
        OUTPUT_DIR / "static_inceptionresnetv2_merged_df_old.csv"
    )
    dados_regressao_final_old = pd.read_csv(caminho_estatico)

    for numero_temporal in range(1, 101):
        caminho_feature = (
            EXECUCAO_LOOP_OLD_DIR
            / f"feature_{numero_temporal:03d}.npy"
        )
        if not caminho_feature.is_file():
            raise FileNotFoundError(
                f"Feature não encontrada: {caminho_feature}"
            )
        dados_regressao_final_old[str(1535 + numero_temporal)] = (
            np.load(caminho_feature)
        )

features_finais_old = [str(i) for i in range(1636)]
colunas_metadados_finais_old = [
    "y",
    "current",
    "nozzle_speed",
    "mass_flow_rate",
    "experimento",
]

if dados_regressao_final_old.shape != (26634, 1641):
    raise ValueError(
        f"Formato inesperado: {dados_regressao_final_old.shape}"
    )

if (
    dados_regressao_final_old.drop(
        columns=colunas_metadados_finais_old
    ).columns.tolist()
    != features_finais_old
):
    raise ValueError(
        "As colunas de entrada não correspondem às features 0 a 1635."
    )

X_final_old = dados_regressao_final_old[features_finais_old].astype(
    np.float32
)
y_final_old = dados_regressao_final_old["y"].astype(np.float32)

if not np.isfinite(X_final_old.to_numpy()).all():
    raise ValueError("Há NaN ou infinitos nas entradas.")
if not np.isfinite(y_final_old.to_numpy()).all():
    raise ValueError("Há NaN ou infinitos no alvo.")

tf.random.set_seed(42)

X_train_final_old, X_val_final_old, y_train_final_old, y_val_final_old = (
    train_test_split(
        X_final_old,
        y_final_old,
        train_size=0.7,
        random_state=42,
    )
)

modelo_final_100_old = keras.Sequential([
    keras.layers.Input(shape=(X_train_final_old.shape[1],)),
    keras.layers.Dense(32, activation="relu"),
    keras.layers.Dense(1),
])

early_stopping_final_old = keras.callbacks.EarlyStopping(
    monitor="val_loss",
    patience=20,
    restore_best_weights=True,
)

modelo_final_100_old.compile(
    optimizer="adam",
    loss="mean_absolute_error",
)

print(
    f"Entrada: {X_train_final_old.shape[1]} features "
    "(1.536 visuais + 100 temporais)"
)
print(
    f"Treino: {X_train_final_old.shape} | "
    f"Validação: {X_val_final_old.shape}"
)

inicio_treino_final_old = perf_counter()
historico_final_100_old = modelo_final_100_old.fit(
    X_train_final_old,
    y_train_final_old,
    epochs=300,
    batch_size=128,
    validation_data=(X_val_final_old, y_val_final_old),
    callbacks=[early_stopping_final_old],
    verbose=2,
)
tempo_treino_final_old = perf_counter() - inicio_treino_final_old

y_pred_final_old = modelo_final_100_old.predict(
    X_val_final_old,
    verbose=0,
).reshape(-1)

metricas_final_100_old_df = pd.DataFrame([{
    "metodologia": "old",
    "features_temporais": 100,
    "particao": "validacao",
    "n_features_entrada": X_train_final_old.shape[1],
    "n_treino": len(y_train_final_old),
    "n_validacao": len(y_val_final_old),
    "MAE_mm": mean_absolute_error(
        y_val_final_old, y_pred_final_old
    ),
    "MAPE_pct": 100 * mean_absolute_percentage_error(
        y_val_final_old, y_pred_final_old
    ),
    "R2": r2_score(y_val_final_old, y_pred_final_old),
    "epocas_executadas": len(
        historico_final_100_old.history["loss"]
    ),
    "melhor_epoca": int(
        np.argmin(
            historico_final_100_old.history["val_loss"]
        ) + 1
    ),
    "tempo_treino_s": tempo_treino_final_old,
}])

previsoes_final_100_old_df = pd.DataFrame({
    "linha_dataset": y_val_final_old.index.to_numpy(),
    "experimento": dados_regressao_final_old.loc[
        y_val_final_old.index, "experimento"
    ].to_numpy(),
    "y": y_val_final_old.to_numpy(),
    "y_pred": y_pred_final_old,
})
previsoes_final_100_old_df["erro_absoluto_mm"] = np.abs(
    previsoes_final_100_old_df["y"]
    - previsoes_final_100_old_df["y_pred"]
)

identificador_final_old = datetime.now(
    timezone.utc
).strftime("%Y%m%d_%H%M%S_%f_UTC")

EXECUCAO_REGRESSAO_FINAL_OLD_DIR = (
    EXECUCAO_LOOP_OLD_DIR
    / "regressao_final_100_temporais_old"
    / identificador_final_old
)
EXECUCAO_REGRESSAO_FINAL_OLD_DIR.mkdir(
    parents=True,
    exist_ok=False,
)

modelo_final_100_old.save(
    EXECUCAO_REGRESSAO_FINAL_OLD_DIR
    / "modelo_final_100_temporais_old.keras"
)
metricas_final_100_old_df.to_csv(
    EXECUCAO_REGRESSAO_FINAL_OLD_DIR
    / "metricas_final_100_temporais_old.csv",
    index=False,
)
previsoes_final_100_old_df.to_csv(
    EXECUCAO_REGRESSAO_FINAL_OLD_DIR
    / "previsoes_final_100_temporais_old.csv",
    index=False,
)
pd.DataFrame(
    historico_final_100_old.history
).rename_axis("epoca_base_zero").to_csv(
    EXECUCAO_REGRESSAO_FINAL_OLD_DIR
    / "historico_final_100_temporais_old.csv"
)
pd.DataFrame({
    "linha_dataset": y_train_final_old.index.to_numpy()
}).to_csv(
    EXECUCAO_REGRESSAO_FINAL_OLD_DIR
    / "indices_treino_final_100_temporais_old.csv",
    index=False,
)

display(metricas_final_100_old_df)
display(previsoes_final_100_old_df.head())
print(
    f"Resultados salvos em: "
    f"{EXECUCAO_REGRESSAO_FINAL_OLD_DIR}"
)


In [ ]:
# Resultados gerais — comparação centrada no extrator CNN
from pathlib import Path
from time import perf_counter
from sklearn.metrics import (
    mean_absolute_error,
    mean_absolute_percentage_error,
    mean_squared_error,
    r2_score,
)

# O mesmo lote e o mesmo procedimento deverão ser usados nas outras CNNs.
TAMANHO_LOTE_BENCHMARK = 30
REPETICOES_BENCHMARK = 3

if "y_val_final_old" not in globals() or "y_pred_final_old" not in globals():
    raise RuntimeError("Execute primeiro a regressão final com 100 temporais.")

if "extrator_inceptionresnetv2" in globals():
    extrator_benchmark_old = extrator_inceptionresnetv2
else:
    # Recupera a arquitetura usada na extração sem refazer o dataset.
    extrator_benchmark_old = keras.applications.InceptionResNetV2(
        weights="imagenet", include_top=False, pooling="max"
    )
    extrator_benchmark_old.trainable = False

raiz_imagens_old = Path(globals().get(
    "DATASET_ROOT",
    "/content/tcc_inceptionresnetv2/experimentos_taguchi_400p_cropped_images",
))
caminhos_benchmark_old = sorted(
    (raiz_imagens_old / "L1").glob("*.png"),
    key=lambda caminho: int(caminho.stem),
)[:TAMANHO_LOTE_BENCHMARK]
if len(caminhos_benchmark_old) != TAMANHO_LOTE_BENCHMARK:
    raise FileNotFoundError("Não encontrei as 30 imagens de L1 para o benchmark.")

# Leitura e pré-processamento ficam fora da medição: o tempo reflete
# a passagem do lote pela CNN, via model.predict().
lote_imagens_old = np.stack([
    keras.utils.img_to_array(keras.utils.load_img(caminho))
    for caminho in caminhos_benchmark_old
]).astype(np.float32)
lote_preprocessado_old = (
    keras.applications.inception_resnet_v2.preprocess_input(
        lote_imagens_old
    )
)

# Uma execução inicial evita que a inicialização da rede entre na medida.
extrator_benchmark_old.predict(lote_preprocessado_old, verbose=0)
inicio_benchmark_old = perf_counter()
for _ in range(REPETICOES_BENCHMARK):
    extrator_benchmark_old.predict(lote_preprocessado_old, verbose=0)
tempo_benchmark_old_s = perf_counter() - inicio_benchmark_old
imagens_processadas_benchmark_old = (
    TAMANHO_LOTE_BENCHMARK * REPETICOES_BENCHMARK
)
tempo_inferencia_ms_imagem_old = (
    1000 * tempo_benchmark_old_s / imagens_processadas_benchmark_old
)
vazao_imagens_s_old = (
    imagens_processadas_benchmark_old / tempo_benchmark_old_s
)

gpus_benchmark_old = tf.config.list_physical_devices("GPU")
if gpus_benchmark_old:
    detalhes_gpu_old = tf.config.experimental.get_device_details(
        gpus_benchmark_old[0]
    )
    hardware_benchmark_old = detalhes_gpu_old.get("device_name", "GPU")
else:
    hardware_benchmark_old = "CPU"

n_parametros_extrator_old = int(extrator_benchmark_old.count_params())
n_parametros_treinaveis_old = int(sum(
    np.prod(peso.shape)
    for peso in extrator_benchmark_old.trainable_weights
))
n_features_visuais_old = int(extrator_benchmark_old.output_shape[-1])
n_features_temporais_old = int(
    X_val_final_old.shape[1] - n_features_visuais_old
)

COLUNAS_COMPARACAO_CNN = [
    "metodologia", "extrator", "pesos", "entrada_imagem",
    "preprocessamento", "pooling", "congelado",
    "parametros_milhoes", "parametros_treinaveis",
    "peso_teorico_fp32_MiB", "features_visuais",
    "imagens_no_dataset", "tempo_inferencia_ms_imagem",
    "vazao_imagens_s", "hardware_benchmark",
    "lote_benchmark", "repeticoes_benchmark",
    "features_temporais", "regressor", "particao",
    "n_treino", "n_validacao", "MAE_mm", "MAPE_pct",
    "RMSE_mm", "R2",
]

linha_comparacao_old = {
    "metodologia": "old",
    "extrator": "InceptionResNetV2",
    "pesos": "ImageNet",
    "entrada_imagem": "400x400x3",
    "preprocessamento": "inception_resnet_v2.preprocess_input",
    "pooling": "global max",
    "congelado": not extrator_benchmark_old.trainable,
    "parametros_milhoes": n_parametros_extrator_old / 1e6,
    "parametros_treinaveis": n_parametros_treinaveis_old,
    "peso_teorico_fp32_MiB": (
        4 * n_parametros_extrator_old / (1024 ** 2)
    ),
    "features_visuais": n_features_visuais_old,
    "imagens_no_dataset": len(dados_regressao_final_old),
    "tempo_inferencia_ms_imagem": tempo_inferencia_ms_imagem_old,
    "vazao_imagens_s": vazao_imagens_s_old,
    "hardware_benchmark": hardware_benchmark_old,
    "lote_benchmark": TAMANHO_LOTE_BENCHMARK,
    "repeticoes_benchmark": REPETICOES_BENCHMARK,
    "features_temporais": n_features_temporais_old,
    "regressor": "Dense(32)-Dense(1)",
    "particao": "validacao",
    "n_treino": len(y_train_final_old),
    "n_validacao": len(y_val_final_old),
    "MAE_mm": mean_absolute_error(y_val_final_old, y_pred_final_old),
    "MAPE_pct": 100 * mean_absolute_percentage_error(
        y_val_final_old, y_pred_final_old
    ),
    "RMSE_mm": np.sqrt(mean_squared_error(
        y_val_final_old, y_pred_final_old
    )),
    "R2": r2_score(y_val_final_old, y_pred_final_old),
}
resultados_gerais_old_df = pd.DataFrame(
    [linha_comparacao_old], columns=COLUNAS_COMPARACAO_CNN
)

RESULTADOS_GERAIS_OLD_PATH = (
    Path(EXECUCAO_REGRESSAO_FINAL_OLD_DIR)
    / "resultados_gerais_padronizados_old.csv"
)
resultados_gerais_old_df.to_csv(RESULTADOS_GERAIS_OLD_PATH, index=False)

pd.set_option("display.max_columns", None)
display(resultados_gerais_old_df.round(4))
print(f"Tabela salva em: {RESULTADOS_GERAIS_OLD_PATH}")
print(
    "Tempo de inferência: benchmark de 30 imagens em lote, "
    "repetido 3 vezes; exclui leitura e pré-processamento."
)
